# Exercise-like episode detection in AI-READI (v2 -- single high-precision class)

This notebook detects exercise-like episodes from wearable physiology only.
Signals used: heart rate, accelerometer-derived activity (steps, stage flags),
respiratory rate, and sleep/awake state.

**Signal independence rule:**
- `calories_per_min` and `activity_intensity_score` are descriptive only.
  They are forbidden from every detector gate.
- `cgm_glucose_mean` is loaded for QC diagnostics and plotting only,
  clearly labeled "context only, not used for detection."
  It must not influence any label.

**What changed from v1:**
- Resting HR baseline tightened from 20th to 5th percentile, restricted to
  sedentary-flagged rows where available.
- Four-class detector collapsed to one: `ambulatory_exercise_like`.
  Gate requires BOTH `HR_local_delta_10min >= 10` AND `steps_mean_10 >= 60`.
- `uncertain_arousal` retained as an audit-only reject bin, never merged with
  the exercise class.
- Merge gap shortened from 10 to 5 min. Episodes over 90 min are split at the
  HR trough. Minimum episode length is 10 min.
- Recovery-decay filter added: episodes are flagged `recovery_pass = False`
  when post-episode HR does not decay as expected.
- QC diagnostics added: meal-confound fraction, episodes-per-active-day,
  RR missingness rate.


## 1. Imports and configuration

In [32]:
import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import pyarrow
    HAS_PYARROW = True
except ImportError:
    HAS_PYARROW = False

try:
    from scipy import stats as sp_stats
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

print(f'pyarrow: {HAS_PYARROW} | scipy: {HAS_SCIPY}')


pyarrow: True | scipy: True


In [33]:
import datetime as _dt
RUN_ID = _dt.datetime.now().strftime('%Y%m%d_%H%M%S')

# ---- paths ----
DATA_PATH   = None   # parquet; None = auto-search
COHORT_PATH = '/home/myriamcharfeddine/CGM/Data/enriched_multimodal/cohort.csv'

OUTPUT_DIR = Path('outputs/exercise_episode_detection_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(exist_ok=True)

# ---- rolling windows (minutes) ----
WIN5, WIN10, WIN15, WIN30 = 5, 10, 15, 30

# ---- segment rules ----
GAP_THRESHOLD_MIN = 5

# ---- stable resting baseline (DETECTION floor: 5th pct) ----
HR_BASELINE_PERCENTILE = 0.05    # 5th pct of sedentary awake rows (whole record)
RR_BASELINE_PERCENTILE = 0.05
MIN_BASELINE_ROWS      = 30      # else cohort fallback

# ---- strain grading baseline (separate from detection) ----
HR_STRAIN_PERCENTILE   = 0.20   # 20th pct -- more robust for personal cost grading

# ---- local rolling baseline: plot / onset tiebreaker only, NEVER in gate ----
LOCAL_LOOKBACK_MIN = 30
LOCAL_LAG_MIN      = 5

# ---- inclusive detection gate ----
STEP_RATE_FLOOR  = 40    # steps/min, 10-min mean (LOW: stroll through run all register)
HR_EXCESS_FLOOR  = 8     # bpm above stable resting floor, 10-min mean

# ---- episode construction ----
ANCHOR_ON_MIN  = 8    # positive minutes in any 10 to start
ANCHOR_OFF_MIN = 5    # consecutive negative to end
MERGE_GAP_MIN  = 5    # max gap to merge two bouts
MAX_BOUT_MIN   = 120  # split longer bouts at steps trough; 7067 walks run ~2h
MIN_BOUT_MIN   = 10   # drop shorter bouts
BACKTRACK_MAX_MIN = 45  # max look-back for refined_start_time onset

# ---- two-axis grading (descriptive, never gates detection) ----
# Axis 1: cadence_class -- activity MODALITY, graded from ACTIVE-CORE minutes only
CADENCE_CORE_STEP_FLOOR = 20   # min steps_mean_5 (or activity_steps_per_min) to be active-core
CADENCE_SLOW_WALK  = 40    # 40-80 steps/min (active-core cadence)
CADENCE_WALK       = 80    # 80-110
CADENCE_BRISK_WALK = 110   # 110-130
CADENCE_RUN_LIKE   = 130   # 130+  (high-cadence ambulation; wrist device, not confirmed running)

# Axis 2: strain_class -- PERSONAL cost, via HR_excess_strain (20th pct floor, not 5th)
STRAIN_LIGHT    = 8    # 8-15 bpm excess
STRAIN_MODERATE = 15   # 15-30
STRAIN_VIGOROUS = 30   # 30+

# ---- QC thresholds ----
MAX_SLEEP_FRAC         = 0.05
RECOVERY_MIN           = 30     # kept for compat; fast path uses RECOVERY_WINDOW_MIN
RECOVERY_WINDOW_MIN    = 60     # 60-min window; intense bouts need > 30 min
RECOVERY_REST_STEPS    = 10     # only resting minutes count (steps <= this)
RECOVERY_FRAC_THRESH   = 0.30
GLUCOSE_RISE_THRESH_MG = 20
LONG_ACTIVE_HOURS      = 3.0   # long-active-envelope flag threshold

print(f'Configuration loaded.  RUN_ID={RUN_ID}')


Configuration loaded.  RUN_ID=20260628_185301


## 2. Locate and load data

In [34]:
LOAD_COLS = [
    'participant_id', 'timestamp_local',
    'heart_rate_mean', 'heart_rate_min', 'heart_rate_max', 'heart_rate_std',
    'activity_steps_per_min',
    'activity_stage_walking', 'activity_stage_running',
    'activity_stage_sedentary', 'activity_stage_generic',
    'activity_transitions', 'activity_intensity_score',
    'calories_per_min', 'calories_total',
    'respiratory_rate_mean',
    'stress_level_mean',
    'sleep_stage_awake', 'sleep_stage_deep',
    'sleep_stage_light', 'sleep_stage_rem', 'sleep_stage_unknown',
    'cgm_glucose_mean',
]

REQUIRED_COLS = {'participant_id', 'timestamp_local', 'heart_rate_mean', 'activity_steps_per_min'}
PREFERRED_PATH_FRAGMENTS = ['enriched_multimodal', 'final_multimodal', 'multimodal_dataset']


def _score_candidate(path):
    path_str = str(path).lower()
    fragment_score = sum(frag in path_str for frag in PREFERRED_PATH_FRAGMENTS)
    penalty = any(x in path_str for x in ['cache', 'forecast', 'benchmark', 'result', 'sweep'])
    return (0 if penalty else 1) + fragment_score


def _has_required_cols(path):
    try:
        import pyarrow.parquet as pq
        return REQUIRED_COLS.issubset(set(pq.read_schema(path).names))
    except Exception:
        return False


def _find_parquet(search_roots):
    candidates = []
    for root in search_roots:
        p = Path(root)
        if not p.exists():
            continue
        for f in p.rglob('*.parquet'):
            try:
                sz = f.stat().st_size
            except OSError:
                continue
            if sz < 1_000_000:
                continue
            candidates.append(f)
    scored = [((-_score_candidate(f), -f.stat().st_size), f) for f in candidates]
    scored.sort(key=lambda x: x[0])
    for _, f in scored:
        if _has_required_cols(f):
            return f
    return None


if DATA_PATH is not None:
    data_path = Path(DATA_PATH)
else:
    search_roots = [
        Path.home() / 'CGM' / 'Data',
        Path.home() / 'CGM' / 'Preprocess',
        Path.home() / 'CGM',
        Path('/data'), Path('/mnt'), Path.cwd().parent,
    ]
    data_path = _find_parquet(search_roots)
    if data_path is None:
        raise FileNotFoundError(
            'Could not auto-locate a valid wearable parquet. Set DATA_PATH explicitly.'
        )
    print(f'Auto-located parquet: {data_path}')


def _read_parquet(path, cols):
    import pyarrow.parquet as pq
    available = set(pq.read_schema(path).names)
    use_cols  = [c for c in cols if c in available]
    missing   = [c for c in cols if c not in available]
    if missing:
        print(f'  Missing columns (set to NaN): {missing}')
    df = pd.read_parquet(path, columns=use_cols)
    for c in missing:
        df[c] = np.nan
    return df


df = _read_parquet(data_path, LOAD_COLS)
print(f'Loaded {len(df):,} rows x {df.shape[1]} columns')


Auto-located parquet: /home/myriamcharfeddine/CGM/Data/enriched_multimodal/final_multimodal_dataset_20260515_184339.parquet
Loaded 4,532,158 rows x 23 columns


## 3. Parse timestamps, sort, deduplicate, assign segment IDs

In [35]:
df['timestamp_local'] = pd.to_datetime(df['timestamp_local'], utc=False)
df = df.sort_values(['participant_id', 'timestamp_local']).reset_index(drop=True)

n_before = len(df)
df = df.drop_duplicates(subset=['participant_id', 'timestamp_local'], keep='first')
print(f'Deduplication removed {n_before - len(df):,} rows')

df['dt_min'] = (
    df.groupby('participant_id')['timestamp_local']
      .diff().dt.total_seconds().div(60)
)

cohort_boundaries = set()
if COHORT_PATH is not None:
    cdf = pd.read_csv(COHORT_PATH)
    if 'participant_id' in cdf.columns and 'timestamp_local' in cdf.columns:
        cdf['timestamp_local'] = pd.to_datetime(cdf['timestamp_local'])
        for pid, ts in zip(cdf['participant_id'], cdf['timestamp_local']):
            cohort_boundaries.add((pid, ts))
        print(f'Loaded {len(cohort_boundaries)} cohort boundaries')

seg_id = np.zeros(len(df), dtype=int)
counter = 0
prev_pid = None
for i, row in enumerate(df.itertuples(index=False)):
    pid = row.participant_id
    dt  = row.dt_min
    ts  = row.timestamp_local
    new_seg = (
        (pid != prev_pid)
        or (pd.notna(dt) and dt > GAP_THRESHOLD_MIN)
        or ((pid, ts) in cohort_boundaries)
    )
    if new_seg:
        counter += 1
    seg_id[i] = counter
    prev_pid = pid

df['segment_id'] = seg_id
print(f'Created {df["segment_id"].nunique():,} segments for {df["participant_id"].nunique():,} participants')


Deduplication removed 0 rows
Created 1,892 segments for 1,892 participants


## 2b. Cohort guard -- restrict to base-model population

In [36]:
# ---- Cohort guard: restrict to the selected cohort before any computation ----
#
# The base model was trained and evaluated on participants that passed the 4-stage
# cohort selection (duration floor -> boundary trim -> gap splitting -> window count).
# Episodes from non-cohort participants have no valid forecast windows and cannot
# be used in the Stage 2 residual diagnostic. Filter here, before baseline and gate.

COHORT_PATH_DEFAULT = Path('/home/myriamcharfeddine/CGM/Data/enriched_multimodal/cohort.csv')
_cohort_path = Path(COHORT_PATH) if COHORT_PATH else COHORT_PATH_DEFAULT

if _cohort_path.exists():
    _cohort = pd.read_csv(_cohort_path)
    # cohort.csv participant_id may be int or str; match df dtype
    _df_pid_dtype = df['participant_id'].dtype
    if pd.api.types.is_integer_dtype(_df_pid_dtype):
        _cohort_pids = set(_cohort['participant_id'].astype(int))
    else:
        _cohort_pids = set(_cohort['participant_id'].astype(str))

    _n_full  = df['participant_id'].nunique()
    _n_rows_full = len(df)

    df = df[df['participant_id'].isin(_cohort_pids)].copy()

    _n_cohort     = df['participant_id'].nunique()
    _n_rows_cohort = len(df)
    _n_dropped     = _n_full - _n_cohort

    print(f'Cohort guard applied: {_cohort_path.name}')
    print(f'  Full parquet:    {_n_full} participants | {_n_rows_full:,} rows')
    print(f'  Selected cohort: {_n_cohort} participants | {_n_rows_cohort:,} rows')
    print(f'  Excluded: {_n_dropped} participants (no valid 48h forecast windows)')
    print(f'  All downstream episode counts reflect the selected cohort only.')
else:
    print(f'[WARN] cohort.csv not found at {_cohort_path}')
    print(f'       Running on full parquet ({df["participant_id"].nunique()} participants).')
    print(f'       Set COHORT_PATH in the config cell to restrict to the base-model cohort.')


Cohort guard applied: cohort.csv
  Full parquet:    1892 participants | 4,532,158 rows
  Selected cohort: 1591 participants | 4,037,026 rows
  Excluded: 301 participants (no valid 48h forecast windows)
  All downstream episode counts reflect the selected cohort only.


## 4. Sleep and awake flags

In [37]:
SLEEP_COLS = ['sleep_stage_deep', 'sleep_stage_light', 'sleep_stage_rem',
              'sleep_stage_awake', 'sleep_stage_unknown']
for c in SLEEP_COLS:
    if c not in df.columns:
        df[c] = np.nan

sleep_sum = df[SLEEP_COLS].fillna(0).sum(axis=1)
df['is_in_sleep_session']  = sleep_sum > 0
df['is_asleep']            = df['is_in_sleep_session'] & (df['sleep_stage_awake'].fillna(0) < 0.5)
df['is_awake_for_exercise'] = ~df['is_asleep']

print(f'Asleep fraction:              {df["is_asleep"].mean():.3f}')
print(f'Awake-for-exercise fraction:  {df["is_awake_for_exercise"].mean():.3f}')


Asleep fraction:              0.254
Awake-for-exercise fraction:  0.746


## 5. Participant-specific HR and RR baselines (5th percentile, sedentary rows)

Baseline rows are restricted to awake, sedentary-flagged minutes when
`activity_stage_sedentary` is available. Resting HR is the 5th percentile
of qualifying rows per participant (tighter than the previous 20th percentile).
Participants with fewer than 30 qualifying rows fall back to the cohort 5th
percentile and are flagged `baseline_low_confidence = True`.


In [38]:
def _select_baseline_rows(grp):
    """Return sedentary+awake rows for the stable resting floor (whole record)."""
    awake = grp[grp['is_awake_for_exercise']]
    # prefer sedentary stage; fall back to steps<=5 if stage column is weak
    if 'activity_stage_sedentary' in awake.columns:
        sed = awake[awake['activity_stage_sedentary'].fillna(0) > 0.5]
        if len(sed) >= MIN_BASELINE_ROWS:
            return sed
    low_step = awake[awake['activity_steps_per_min'].fillna(999) <= 5]
    if len(low_step) >= MIN_BASELINE_ROWS:
        return low_step
    return awake


hr_valid_awake = df[df['is_awake_for_exercise'] & df['heart_rate_mean'].notna()]
COHORT_HR_REST        = hr_valid_awake['heart_rate_mean'].quantile(HR_BASELINE_PERCENTILE)
COHORT_HR_REST_STRAIN = hr_valid_awake['heart_rate_mean'].quantile(HR_STRAIN_PERCENTILE)
COHORT_HR_SD          = hr_valid_awake['heart_rate_mean'].std()
rr_valid_awake = df[df['is_awake_for_exercise'] & df['respiratory_rate_mean'].notna()]
COHORT_RR_REST = rr_valid_awake['respiratory_rate_mean'].quantile(RR_BASELINE_PERCENTILE) if len(rr_valid_awake) else np.nan
COHORT_RR_SD   = rr_valid_awake['respiratory_rate_mean'].std() if len(rr_valid_awake) else 1.0

print(f'Cohort fallback HR_rest_stable={COHORT_HR_REST:.1f}  HR_rest_strain={COHORT_HR_REST_STRAIN:.1f}  HR_sd={COHORT_HR_SD:.1f}')

baselines = {}
low_conf_pids = set()

for pid, grp in df.groupby('participant_id'):
    qual    = _select_baseline_rows(grp)
    hr_rows = qual['heart_rate_mean'].dropna()
    rr_rows = qual['respiratory_rate_mean'].dropna()

    low_conf = len(hr_rows) < MIN_BASELINE_ROWS
    if low_conf:
        low_conf_pids.add(pid)

    hr_rest        = hr_rows.quantile(HR_BASELINE_PERCENTILE) if len(hr_rows) >= 5 else COHORT_HR_REST
    hr_rest_strain = hr_rows.quantile(HR_STRAIN_PERCENTILE)   if len(hr_rows) >= 5 else COHORT_HR_REST_STRAIN
    hr_sd          = hr_rows.std()                            if len(hr_rows) >= 5 else COHORT_HR_SD
    rr_rest = rr_rows.quantile(RR_BASELINE_PERCENTILE) if len(rr_rows) >= 5 else COHORT_RR_REST
    rr_sd   = rr_rows.std()                            if len(rr_rows) >= 5 else COHORT_RR_SD

    baselines[pid] = dict(
        HR_rest_stable=hr_rest, HR_rest_strain=hr_rest_strain, HR_sd=max(hr_sd, 1.0),
        RR_rest=rr_rest,        RR_sd=max(rr_sd, 0.5),
        low_confidence=low_conf,
    )

bl_df = (pd.DataFrame(baselines).T
           .reset_index()
           .rename(columns={'index': 'participant_id'}))
df = df.merge(bl_df, on='participant_id', how='left')

# Detection floor: 5th pct (sensitive, for gate)
df['HR_excess_stable'] = df['heart_rate_mean'] - df['HR_rest_stable']
# Strain grading floor: 20th pct (more robust, for grading only, never gates)
df['HR_excess_strain'] = df['heart_rate_mean'] - df['HR_rest_strain']
df['HR_z']             = df['HR_excess_stable'] / df['HR_sd']
df['RR_excess']        = df['respiratory_rate_mean'] - df['RR_rest']
df['RR_z']             = df['RR_excess'] / df['RR_sd']

# Backward compat aliases
df['HR_rest']   = df['HR_rest_stable']
df['HR_excess'] = df['HR_excess_stable']

n_low = len(low_conf_pids)
n_tot = df['participant_id'].nunique()
print(f'Stable baseline: {n_tot - n_low}/{n_tot} participants on personal floor, '
      f'{n_low} on cohort fallback (baseline_low_confidence=True).')
bl_vals = bl_df['HR_rest_stable'].dropna()
print(f'HR_rest_stable (detection, 5th pct):  '
      f'p10={bl_vals.quantile(0.10):.1f}  '
      f'median={bl_vals.median():.1f}  '
      f'p90={bl_vals.quantile(0.90):.1f}  '
      f'range=[{bl_vals.min():.1f}, {bl_vals.max():.1f}] bpm')
bl_strain = bl_df['HR_rest_strain'].dropna()
print(f'HR_rest_strain  (grading,    20th pct):  '
      f'p10={bl_strain.quantile(0.10):.1f}  '
      f'median={bl_strain.median():.1f}  '
      f'p90={bl_strain.quantile(0.90):.1f}  '
      f'range=[{bl_strain.min():.1f}, {bl_strain.max():.1f}] bpm')
extreme_hi = bl_vals[bl_vals > 85]
extreme_lo = bl_vals[bl_vals < 45]
if len(extreme_hi):
    print(f'[WARN] {len(extreme_hi)} participants with HR_rest_stable > 85 bpm -- inspect sedentary selection.')
if len(extreme_lo):
    print(f'[INFO] {len(extreme_lo)} participants with HR_rest_stable < 45 bpm -- likely athletes.')


Cohort fallback HR_rest_stable=57.0  HR_rest_strain=66.0  HR_sd=14.8
Stable baseline: 1591/1591 participants on personal floor, 0 on cohort fallback (baseline_low_confidence=True).
HR_rest_stable (detection, 5th pct):  p10=52.0  median=62.0  p90=74.0  range=[35.0, 92.2] bpm
HR_rest_strain  (grading,    20th pct):  p10=57.0  median=67.5  p90=79.2  range=[38.4, 94.5] bpm
[WARN] 7 participants with HR_rest_stable > 85 bpm -- inspect sedentary selection.
[INFO] 16 participants with HR_rest_stable < 45 bpm -- likely athletes.


### 5b. Local delta features

`HR_local_delta` = heart_rate_mean minus the median of the 25 minutes ending
5 minutes ago (t-30 to t-5, past only, within segment). This captures intra-day
HR onset relative to recent sedentary context, which is more precise than the
global resting baseline for bouts that start during an already-active day.

**Key constraint:** delta HR alone never creates a label. The steps gate is
mandatory and must satisfy >= 60 steps/min independently.


In [39]:
def seg_shift(df, col, periods):
    result = np.full(len(df), np.nan)
    for seg_id, grp in df.groupby('segment_id', sort=False):
        idx  = grp.index.to_numpy()
        vals = pd.Series(grp[col].to_numpy(dtype=float))
        result[idx] = vals.shift(periods).to_numpy()
    return result


def local_rolling_baseline(df, col, lookback=LOCAL_LOOKBACK_MIN, lag=LOCAL_LAG_MIN):
    window = lookback - lag
    result = np.full(len(df), np.nan)
    for seg_id, grp in df.groupby('segment_id', sort=False):
        idx = grp.index.to_numpy()
        s   = pd.Series(grp[col].to_numpy(dtype=float)).shift(lag)
        r   = s.rolling(window, min_periods=max(1, window // 2)).median()
        result[idx] = r.to_numpy()
    return result


df['HR_local_baseline_30'] = local_rolling_baseline(df, 'heart_rate_mean')
df['RR_local_baseline_30'] = local_rolling_baseline(df, 'respiratory_rate_mean')
df['HR_local_delta'] = df['heart_rate_mean'].to_numpy() - df['HR_local_baseline_30'].to_numpy()
df['RR_local_delta'] = df['respiratory_rate_mean'].to_numpy() - df['RR_local_baseline_30'].to_numpy()

print(f'HR_local_delta non-null: {df["HR_local_delta"].notna().sum():,}')


HR_local_delta non-null: 3,737,015


## 6. Rolling features

In [40]:
def rolling_seg(df, col, window, agg='mean', fill=np.nan):
    result = np.full(len(df), fill, dtype=float)
    for seg_id, grp in df.groupby('segment_id', sort=False):
        idx  = grp.index
        vals = grp[col].to_numpy(dtype=float)
        s    = pd.Series(vals)
        if agg == 'mean':
            r = s.rolling(window, min_periods=max(1, window // 2)).mean()
        elif agg == 'max':
            r = s.rolling(window, min_periods=max(1, window // 2)).max()
        elif agg == 'sum':
            r = s.rolling(window, min_periods=max(1, window // 2)).sum()
        result[idx] = r.to_numpy()
    return result


# ---- step/activity rolling features (lead detection, HR-independent) ----
df['steps_mean_5']  = rolling_seg(df, 'activity_steps_per_min', WIN5)
df['steps_mean_10'] = rolling_seg(df, 'activity_steps_per_min', WIN10)
df['steps_mean_15'] = rolling_seg(df, 'activity_steps_per_min', WIN15)

df['_active_raw'] = (
    df['activity_stage_walking'].fillna(0)
    + df['activity_stage_running'].fillna(0)
    + df['activity_stage_generic'].fillna(0)
)
df['active_fraction_10'] = rolling_seg(df, '_active_raw', WIN10)
df['active_fraction_5']  = rolling_seg(df, '_active_raw', WIN5)

# ---- stable-floor excess rolling means (used in gate and features) ----
df['HR_excess_stable_mean_5']  = rolling_seg(df, 'HR_excess_stable', WIN5)
df['HR_excess_stable_mean_10'] = rolling_seg(df, 'HR_excess_stable', WIN10)
df['HR_z_mean_10']             = rolling_seg(df, 'HR_z',             WIN10)

# ---- RR (descriptive only; device does not track onset reliably) ----
df['RR_excess_mean_10']        = rolling_seg(df, 'RR_excess', WIN10)

# ---- HRLoad for episode feature computation ----
df['_hr_excess_pos']    = df['HR_excess_stable'].clip(lower=0)
df['HRLoad_rolling_15'] = rolling_seg(df, '_hr_excess_pos', WIN15, agg='sum')

# ---- local rolling baseline (plot / onset tiebreaker ONLY -- never in gate) ----
# Computed here so figures can show it as a diagnostic trace.
df['HR_local_delta']         = rolling_seg(df, 'HR_excess_stable', WIN5)   # alias for compat
df['HR_local_delta_mean_10'] = rolling_seg(df, 'HR_excess_stable', WIN10)  # plot only
df['HR_local_delta_mean_5']  = rolling_seg(df, 'HR_excess_stable', WIN5)   # plot only

# ---- 5-min onset refinement features ----
df['walking_mean_5']    = rolling_seg(df, 'activity_stage_walking',  WIN5)
df['running_mean_5']    = rolling_seg(df, 'activity_stage_running',  WIN5)
df['generic_mean_5']    = rolling_seg(df, 'activity_stage_generic',  WIN5)
df['transitions_sum_5'] = rolling_seg(df, 'activity_transitions',    WIN5, agg='sum')

# ---- descriptive-only (forbidden from gates) ----
df['activity_intensity_mean_10'] = rolling_seg(df, 'activity_intensity_score', WIN10)
df['calories_per_min_mean_10']   = rolling_seg(df, 'calories_per_min',         WIN10)

print('Rolling features done.')
print(f'  HR_excess_stable_mean_10 non-null: {df["HR_excess_stable_mean_10"].notna().sum():,}')
print(f'  steps_mean_10 non-null:            {df["steps_mean_10"].notna().sum():,}')


Rolling features done.
  HR_excess_stable_mean_10 non-null: 3,892,244
  steps_mean_10 non-null:            4,023,294


In [41]:
# ---- Extra 5-min rolling features for onset refinement ----
# Steps, stage fractions, and transitions at 5-min resolution.
# Used only for backtracking activity onset -- not in detector gates.

WIN5R = 5   # alias to make intent clear

df['steps_mean_5']       = rolling_seg(df, 'activity_steps_per_min', WIN5R)
df['walking_mean_5']     = rolling_seg(df, 'activity_stage_walking',   WIN5R)
df['running_mean_5']     = rolling_seg(df, 'activity_stage_running',   WIN5R)
df['generic_mean_5']     = rolling_seg(df, 'activity_stage_generic',   WIN5R)
df['transitions_sum_5']  = rolling_seg(df, 'activity_transitions',     WIN5R, agg='sum')

df['_active5_raw'] = (
    df['activity_stage_walking'].fillna(0)
    + df['activity_stage_running'].fillna(0)
    + df['activity_stage_generic'].fillna(0)
)
df['active_fraction_5'] = rolling_seg(df, '_active5_raw', WIN5R)

# ---- Per-participant onset percentile thresholds ----
# steps_p70 and transitions_p70/p60 of awake awake rows.
# Used for personalized activity onset candidates.
onset_pct_rows = []
for pid, grp in df.groupby('participant_id'):
    awake = grp[grp['is_awake_for_exercise']]
    st    = awake['steps_mean_5'].dropna()
    tr    = awake['transitions_sum_5'].dropna()
    onset_pct_rows.append(dict(
        participant_id      = pid,
        steps_p70_onset     = st.quantile(0.70) if len(st) >= 5 else 20.0,
        transitions_p70_onset = tr.quantile(0.70) if len(tr) >= 5 else np.nan,
        transitions_p60_onset = tr.quantile(0.60) if len(tr) >= 5 else np.nan,
    ))

onset_pct_df = pd.DataFrame(onset_pct_rows)
df = df.merge(onset_pct_df, on='participant_id', how='left')

print('5-min onset rolling features added.')
print(f'  steps_mean_5 non-null:       {df["steps_mean_5"].notna().sum():,}')
print(f'  active_fraction_5 non-null:  {df["active_fraction_5"].notna().sum():,}')


5-min onset rolling features added.
  steps_mean_5 non-null:       4,028,119
  active_fraction_5 non-null:  4,035,435


## PAUSE 1 -- Baseline and data summary (review before proceeding)


In [42]:
print('=' * 60)
print('PAUSE 1: Stable baseline and gate summary')
print('=' * 60)

print(f'  Participants total:           {df["participant_id"].nunique()}')
print(f'  On personal HR floor:         {df["participant_id"].nunique() - len(low_conf_pids)}')
print(f'  On cohort fallback:           {len(low_conf_pids)}')

if 'HR_rest_stable' in df.columns:
    bl_summary = (df.drop_duplicates('participant_id')
                    .set_index('participant_id')['HR_rest_stable']
                    .dropna())
    print(f'  HR_rest_stable: p10={bl_summary.quantile(0.1):.1f}  '
          f'median={bl_summary.median():.1f}  '
          f'p90={bl_summary.quantile(0.9):.1f} bpm')

print(f'  Gate (applied in next cell): steps_mean_10 >= {STEP_RATE_FLOOR} AND '
      f'HR_excess_stable_mean_10 >= {HR_EXCESS_FLOOR}')
print()
print('Review before continuing to episode construction.')
print('=' * 60)


PAUSE 1: Stable baseline and gate summary
  Participants total:           1591
  On personal HR floor:         1591
  On cohort fallback:           0
  HR_rest_stable: p10=52.0  median=62.0  p90=74.0 bpm
  Gate (applied in next cell): steps_mean_10 >= 40 AND HR_excess_stable_mean_10 >= 8

Review before continuing to episode construction.


## 7. Single high-precision gate -- ambulatory_exercise_like

An anchor minute fires when ALL four conditions hold:
1. `is_awake_for_exercise`
2. `heart_rate_mean` is valid and positive
3. `activity_steps_per_min` is valid
4. `HR_local_delta_10min >= 10` bpm (genuine HR onset above local context)
5. `steps_mean_10 >= 60` steps/min (confirmed ambulation; mechanically independent of HR)

`uncertain_arousal` is an audit-only reject bin (HR/RR elevated, steps low).
It is never merged with or promoted to the exercise class.

`calories_per_min` and `activity_intensity_score` are NOT used in any gate.


In [43]:
# ---- Inclusive detection gate ----
#
# Condition A: sustained step rate >= STEP_RATE_FLOOR (leads detection; HR-independent)
# Condition B: sustained HR cost >= HR_EXCESS_FLOOR against the STABLE resting floor
# Condition C: awake
#
# Floor is LOW (40 steps/min) so slow walks and brisk walks all register.
# Cadence and strain grading happen AFTER detection, not here.
# Rolling local baseline is NOT used as a gate condition.

step_valid = df['activity_steps_per_min'].notna()
hr_valid   = df['heart_rate_mean'].notna() & (df['heart_rate_mean'] > 0)

df['anchor_ambulatory'] = (
    df['is_awake_for_exercise']
    & step_valid
    & hr_valid
    & (df['steps_mean_10'].fillna(0)             >= STEP_RATE_FLOOR)
    & (df['HR_excess_stable_mean_10'].fillna(-999) >= HR_EXCESS_FLOOR)
).astype(int)

# ---- per-row gate failure reason (for debugging; never enters any label) ----
def _gate_failure(row):
    if row['anchor_ambulatory'] == 1:
        return 'passed'
    if not row['is_awake_for_exercise']:
        return 'sleep_failed'
    if not (step_valid.loc[row.name] if hasattr(step_valid, 'loc') else True):
        return 'missing_steps'
    if not (hr_valid.loc[row.name] if hasattr(hr_valid, 'loc') else True):
        return 'missing_hr'
    if row['steps_mean_10'] < STEP_RATE_FLOOR:
        return 'steps_failed'
    if row['HR_excess_stable_mean_10'] < HR_EXCESS_FLOOR:
        return 'HR_failed'
    return 'missing_failed'

# Vectorized failure reason (fast, no per-row apply)
_fr = pd.Series('unknown', index=df.index)
_fr[df['anchor_ambulatory'] == 1]                                    = 'passed'
_passed_mask = df['anchor_ambulatory'] == 1
_fr[~_passed_mask & ~df['is_awake_for_exercise']]                    = 'sleep_failed'
_fr[~_passed_mask & df['is_awake_for_exercise']
    & (df['steps_mean_10'].fillna(0) < STEP_RATE_FLOOR)
    & (df['HR_excess_stable_mean_10'].fillna(-999) < HR_EXCESS_FLOOR)]   = 'steps+HR_failed'
_fr[~_passed_mask & df['is_awake_for_exercise']
    & (df['steps_mean_10'].fillna(0) < STEP_RATE_FLOOR)
    & (df['HR_excess_stable_mean_10'].fillna(-999) >= HR_EXCESS_FLOOR)]  = 'steps_failed'
_fr[~_passed_mask & df['is_awake_for_exercise']
    & (df['steps_mean_10'].fillna(0) >= STEP_RATE_FLOOR)
    & (df['HR_excess_stable_mean_10'].fillna(-999) < HR_EXCESS_FLOOR)]   = 'HR_failed'
_fr[~_passed_mask & df['is_awake_for_exercise']
    & ~step_valid]                                                        = 'missing_steps'
_fr[~_passed_mask & df['is_awake_for_exercise']
    & step_valid & ~hr_valid]                                             = 'missing_hr'
df['gate_failure_reason'] = _fr

# uncertain_arousal: HR elevated but ambulation insufficient (audit-only, not promoted)
hr_elevated = df['HR_excess_stable_mean_10'].fillna(-999) >= HR_EXCESS_FLOOR
low_steps   = df['steps_mean_10'].fillna(999) < STEP_RATE_FLOOR
df['anchor_uncertain_arousal'] = (
    df['is_awake_for_exercise']
    & hr_elevated
    & low_steps
    & (df['anchor_ambulatory'] == 0)
).astype(int)

print(f'Gate: STEP_RATE_FLOOR={STEP_RATE_FLOOR} steps/min, '
      f'HR_EXCESS_FLOOR={HR_EXCESS_FLOOR} bpm (vs stable floor)')
print(f'Anchor ambulatory positive rows:  {df["anchor_ambulatory"].sum():,}')
print(f'Anchor uncertain_arousal rows:    {df["anchor_uncertain_arousal"].sum():,}')
print(f'Gate failure reasons (awake rows):')
awake_df = df[df['is_awake_for_exercise']]
print(awake_df['gate_failure_reason'].value_counts().to_string())


Gate: STEP_RATE_FLOOR=40 steps/min, HR_EXCESS_FLOOR=8 bpm (vs stable floor)
Anchor ambulatory positive rows:  63,665
Anchor uncertain_arousal rows:    2,168,521
Gate failure reasons (awake rows):
gate_failure_reason
steps_failed       2132829
steps+HR_failed     634094
missing_hr          172788
passed               63665
missing_steps         5986
HR_failed              968


## PAUSE 2 -- Anchor-positive summary (review before building episodes)

In [44]:
print('=' * 60)
print('PAUSE 2: Anchor-positive summary (check participant 7067)')
print('=' * 60)

anch = df[df['anchor_ambulatory'] == 1]
per_pid = anch.groupby('participant_id').size().sort_values(ascending=False)
print(f'  Participants with any anchor-positive minute: {len(per_pid)}')
print(f'  Top 5 by anchor count:')
for pid, cnt in per_pid.head(5).items():
    print(f'    {pid}: {cnt} minutes')

pid_7067 = 7067
if pid_7067 in per_pid.index:
    print(f'  Participant {pid_7067}: {per_pid[pid_7067]} anchor-positive minutes')
    # show how many distinct runs (proto-episodes) by looking at gaps
    p_df = df[df["participant_id"] == pid_7067].copy()
    p_anch = p_df["anchor_ambulatory"].to_numpy()
    runs = 0
    in_r = False
    for v in p_anch:
        if v and not in_r:
            runs += 1
            in_r = True
        elif not v:
            in_r = False
    print(f'  Participant {pid_7067}: {runs} anchor-positive run(s) -- '
          f'expect >= 3 for three walks to register.')
    if runs >= 3:
        print(f'  [OK] All three walks register.')
    else:
        print(f'  [WARN] Fewer than 3 runs. Review STEP_RATE_FLOOR or HR_EXCESS_FLOOR.')
else:
    print(f'  Participant {pid_7067} not found in anchor-positive set.')

# ---- Minute-level gate debug for 7067 ----
p7_df = df[df['participant_id'] == pid_7067].copy()
if len(p7_df):
    # Focus on dates where any activity is visible (any steps > 0)
    p7_df['_date'] = p7_df['timestamp_local'].dt.normalize()
    active_dates = p7_df[p7_df['activity_steps_per_min'].fillna(0) > 10]['_date'].unique()
    if len(active_dates):
        debug_date = active_dates[0]  # first active date
        debug_rows = p7_df[p7_df['_date'] == debug_date][
            ['timestamp_local', 'activity_steps_per_min', 'steps_mean_5', 'steps_mean_10',
             'heart_rate_mean', 'HR_rest_stable', 'HR_excess_stable', 'HR_excess_stable_mean_10',
             'is_awake_for_exercise', 'is_asleep', 'anchor_ambulatory', 'gate_failure_reason']
        ]
        print(f'\n  Gate debug -- participant {pid_7067}, first active date={debug_date.strftime("%Y-%m-%d")}')
        print(f'  Rows: {len(debug_rows)}')
        print(f'  Failure reason distribution:')
        print(debug_rows['gate_failure_reason'].value_counts().to_string())
        # Show the active window (steps > 20) in detail
        active_rows = debug_rows[debug_rows['activity_steps_per_min'].fillna(0) > 20]
        if len(active_rows):
            print(f'\n  Minutes with steps > 20 (n={len(active_rows)}):')
            with pd.option_context('display.max_rows', 30, 'display.width', 200,
                                   'display.float_format', lambda x: f'{x:.1f}'):
                print(active_rows.to_string(index=False))
        else:
            print(f'  No minutes with steps > 20 on {debug_date.strftime("%Y-%m-%d")}.')
            print(f'  Top step minutes:')
            print(debug_rows.nlargest(10, 'activity_steps_per_min')[
                ['timestamp_local', 'activity_steps_per_min', 'steps_mean_10',
                 'HR_excess_stable_mean_10', 'is_awake_for_exercise', 'gate_failure_reason']
            ].to_string(index=False))
else:
    print(f'  Participant {pid_7067} not in dataset at all.')

print()
print('Review before continuing to episode construction.')
print('=' * 60)


PAUSE 2: Anchor-positive summary (check participant 7067)
  Participants with any anchor-positive minute: 1187
  Top 5 by anchor count:
    1577: 544 minutes
    4160: 543 minutes
    4472: 526 minutes
    7811: 499 minutes
    7609: 439 minutes
  Participant 7067 not found in anchor-positive set.
  Participant 7067 not in dataset at all.

Review before continuing to episode construction.


## 8. Convert anchors to episodes

In [45]:
def anchors_to_episodes(df, anchor_col, label,
                        on_k=ANCHOR_ON_MIN, off_k=ANCHOR_OFF_MIN):
    episodes = []
    for seg_id, grp in df.groupby('segment_id', sort=False):
        grp   = grp.reset_index()
        pid   = grp['participant_id'].iloc[0]
        flags = grp[anchor_col].to_numpy(dtype=int)
        ts    = grp['timestamp_local'].to_numpy()
        orig  = grp['index'].to_numpy()
        n     = len(flags)
        rc    = pd.Series(flags).rolling(10, min_periods=1).sum().to_numpy()

        in_ep = False; start_i = None; neg_run = 0

        for i in range(n):
            if not in_ep:
                if rc[i] >= on_k:
                    in_ep = True
                    start_i = max(0, i - 9)
                    while start_i < i and flags[start_i] == 0:
                        start_i += 1
                    neg_run = 0
            else:
                if flags[i] == 0:
                    neg_run += 1
                else:
                    neg_run = 0
                if neg_run >= off_k:
                    end_i = i - neg_run
                    episodes.append(dict(
                        participant_id=pid, segment_id=seg_id,
                        confidence_start_time=ts[start_i], end_time=ts[end_i],
                        start_orig=orig[start_i],          end_orig=orig[end_i],
                        detector_label=label))
                    in_ep = False

        if in_ep:
            episodes.append(dict(
                participant_id=pid, segment_id=seg_id,
                confidence_start_time=ts[start_i], end_time=ts[n - 1],
                start_orig=orig[start_i],           end_orig=orig[n - 1],
                detector_label=label))

    return pd.DataFrame(episodes)


def merge_episodes(eps_df, gap_min=MERGE_GAP_MIN):
    if eps_df.empty:
        return eps_df
    merged = []
    for (pid, seg, lbl), grp in eps_df.groupby(
            ['participant_id', 'segment_id', 'detector_label']):
        grp = grp.sort_values('confidence_start_time').reset_index(drop=True)
        cur = grp.iloc[0].to_dict()
        for _, row in grp.iloc[1:].iterrows():
            gap = (pd.Timestamp(row['confidence_start_time'])
                   - pd.Timestamp(cur['end_time'])).total_seconds() / 60
            if gap <= gap_min:
                cur['end_time'] = row['end_time']
                cur['end_orig'] = row['end_orig']
            else:
                merged.append(cur)
                cur = row.to_dict()
        merged.append(cur)
    return pd.DataFrame(merged).reset_index(drop=True)


def split_long_bouts(eps_df, df, max_min=MAX_BOUT_MIN):
    """Split episodes > max_min at the lowest steps_mean_10 trough."""
    out = []
    for _, ep in eps_df.iterrows():
        dur = (pd.Timestamp(ep['end_time'])
               - pd.Timestamp(ep['confidence_start_time'])).total_seconds() / 60
        if dur <= max_min:
            out.append(ep.to_dict())
            continue
        rows = df.iloc[int(ep['start_orig']):int(ep['end_orig']) + 1]
        st   = rows['steps_mean_10'].to_numpy(dtype=float)
        trough_rel = int(np.nanargmin(st))
        trough_i   = rows.index[trough_rel]
        out.append({**ep.to_dict(),
                    'end_time': rows['timestamp_local'].iloc[trough_rel],
                    'end_orig': trough_i})
        out.append({**ep.to_dict(),
                    'confidence_start_time': rows['timestamp_local'].iloc[trough_rel],
                    'start_orig':            trough_i})
    return pd.DataFrame(out).reset_index(drop=True)


def drop_short(eps_df, min_min=MIN_BOUT_MIN):
    eps_df = eps_df.copy()
    eps_df['_dur'] = (
        pd.to_datetime(eps_df['end_time'])
        - pd.to_datetime(eps_df['confidence_start_time'])
    ).dt.total_seconds() / 60
    return eps_df[eps_df['_dur'] >= min_min].drop(columns='_dur').reset_index(drop=True)


def enforce_no_overlap(eps_df):
    out = []; claimed = {}
    for _, ep in eps_df.sort_values('confidence_start_time').iterrows():
        key = (ep['participant_id'], ep['segment_id'])
        if key not in claimed:
            claimed[key] = []
        s, e = int(ep['start_orig']), int(ep['end_orig'])
        if not any(not (e < cs or s > ce) for cs, ce in claimed[key]):
            claimed[key].append((s, e))
            out.append(ep.to_dict())
    return pd.DataFrame(out).reset_index(drop=True)

def compute_refined_starts(eps_df, df):
    """
    Walk backward from confidence_start to where steps_mean_5 first crosses
    STEP_RATE_FLOOR/2, staying within the same segment AND within awake periods.
    Stops at first asleep minute to prevent refined_start entering sleep.
    """
    half_floor = STEP_RATE_FLOOR / 2

    # Pull once into contiguous arrays, indexed by original row position.
    steps5  = df['steps_mean_5'].to_numpy(dtype=float)
    ts      = df['timestamp_local'].to_numpy()
    seg     = df['segment_id'].to_numpy()
    asleep  = df['is_asleep'].fillna(False).to_numpy(dtype=bool)

    # rows-per-minute is 1 here, so BACKTRACK_MAX_MIN maps directly to rows.
    back = int(BACKTRACK_MAX_MIN)

    refined      = []
    refined_orig = []
    for _, ep in eps_df.iterrows():
        c   = int(ep['start_orig'])          # confidence-start row position
        seg_id = seg[c]
        lo  = max(0, c - back)

        # clamp to the same segment so we never cross a gap
        while lo < c and seg[lo] != seg_id:
            lo += 1

        # also clamp past any asleep minute -- refined start must stay awake
        while lo < c and asleep[lo]:
            lo += 1

        s = steps5[lo:c + 1]
        active = np.nan_to_num(s) >= half_floor

        # walk backward from the confidence-start end of the window,
        # stopping at any asleep minute
        idx = len(active) - 1
        while idx > 0 and active[idx - 1]:
            # check the candidate row for sleep
            abs_pos = lo + idx - 1
            if asleep[abs_pos]:
                break
            idx -= 1

        refined.append(pd.Timestamp(ts[lo + idx]))
        refined_orig.append(lo + idx)

    return refined, refined_orig


def compute_refined_ends(eps_df, df):
    """
    Walk forward from confidence_end (last anchor-positive minute) to the last
    minute where steps_mean_5 is at or above STEP_RATE_FLOOR / 2, mirroring
    compute_refined_starts exactly in the forward direction.

    Steps-led only -- no HR condition. Advances while steps >= half_floor;
    stops the first time steps drop below half_floor (same logic as onset
    backtrack, which stops the first time the previous minute is inactive).
    steps_mean_5 carries 5-minute smoothing so a single noisy sample does not
    prematurely truncate the episode.

    Bounds (same as onset):
      1. Same segment -- never cross a gap
      2. Not asleep -- sleep-safe
      3. Refined duration from refined_start <= MAX_BOUT_MIN
      4. Next episode confidence_start - 1 -- no overlap
      5. At most BACKTRACK_MAX_MIN rows past confidence_end
    """
    half_floor  = STEP_RATE_FLOOR / 2

    steps5  = df['steps_mean_5'].to_numpy(dtype=float)
    ts_arr  = df['timestamp_local'].to_numpy()
    seg_arr = df['segment_id'].to_numpy()
    asleep  = df['is_asleep'].fillna(False).to_numpy(dtype=bool)

    import bisect as _bisect
    _seg_starts = {}
    for _, ep in eps_df.iterrows():
        key = (ep['participant_id'], ep['segment_id'])
        _seg_starts.setdefault(key, []).append(int(ep['start_orig']))
    for k in _seg_starts:
        _seg_starts[k].sort()

    refined_ends      = []
    refined_end_origs = []

    for _, ep in eps_df.iterrows():
        e0     = int(ep['end_orig'])
        seg_id = seg_arr[e0]
        t_rs   = pd.Timestamp(ep.get('refined_start_time', ep['confidence_start_time']))

        _starts   = _seg_starts.get((ep['participant_id'], ep['segment_id']), [])
        _bi       = _bisect.bisect_right(_starts, e0)
        nxt_start = _starts[_bi] if _bi < len(_starts) else None

        hi = min(len(steps5) - 1, e0 + int(BACKTRACK_MAX_MIN))
        while hi > e0 and seg_arr[hi] != seg_id:
            hi -= 1
        if nxt_start is not None:
            hi = min(hi, nxt_start - 1)

        # Mirror of onset: advance while next position is still steps-active.
        # Stop the first time steps_mean_5 drops below half_floor -- no
        # confirmation window needed; the 5-min rolling mean already smooths
        # single-sample noise.
        pos = e0
        while pos < hi:
            nxt = pos + 1
            if seg_arr[nxt] != seg_id:
                break
            if asleep[nxt]:
                break
            t_cand = pd.Timestamp(ts_arr[nxt])
            if (t_cand - t_rs).total_seconds() / 60 > MAX_BOUT_MIN:
                break
            if np.nan_to_num(steps5[nxt]) < half_floor:
                break
            pos = nxt

        refined_ends.append(pd.Timestamp(ts_arr[pos]))
        refined_end_origs.append(pos)

    return refined_ends, refined_end_origs


# ---- build pipeline ----
raw_eps = anchors_to_episodes(df, 'anchor_ambulatory', 'ambulatory_exercise_like')
merged  = merge_episodes(raw_eps)
split   = split_long_bouts(merged, df)
trimmed = drop_short(split)
all_eps = enforce_no_overlap(trimmed)

# Item 1+2: refined start (already sleep-safe from compute_refined_starts)
_ref_starts, _ref_start_origs = compute_refined_starts(all_eps, df)
all_eps['refined_start_time'] = _ref_starts
all_eps['start_orig_refined'] = _ref_start_origs
all_eps['start_time']         = all_eps['refined_start_time']

# Item 1: symmetric end-refinement (also sleep-safe via compute_refined_ends)
_ref_ends, _ref_end_origs = compute_refined_ends(all_eps, df)
all_eps['refined_end_time'] = _ref_ends
all_eps['end_orig_refined'] = _ref_end_origs

onset_shifts = (
    pd.to_datetime(all_eps['confidence_start_time'])
    - pd.to_datetime(all_eps['refined_start_time'])
).dt.total_seconds().div(60)
end_shifts = (
    pd.to_datetime(all_eps['refined_end_time'])
    - pd.to_datetime(all_eps['end_time'])
).dt.total_seconds().div(60)
print(f'Onset shift (confidence - refined start): '
      f'mean={onset_shifts.mean():.1f}  '
      f'median={onset_shifts.median():.1f}  '
      f'max={onset_shifts.max():.1f} min')
print(f'End extension (refined end - confidence end): '
      f'mean={end_shifts.mean():.1f}  '
      f'median={end_shifts.median():.1f}  '
      f'max={end_shifts.max():.1f} min')

# Item 2: sleep-safe assertion
_asleep_np = df['is_asleep'].fillna(False).to_numpy(dtype=bool)
_n_start_asleep = int(sum(
    bool(_asleep_np[int(p)]) for p in all_eps['start_orig_refined']
    if 0 <= int(p) < len(_asleep_np)
))
_n_end_asleep = int(sum(
    bool(_asleep_np[int(p)]) for p in all_eps['end_orig_refined']
    if 0 <= int(p) < len(_asleep_np)
))
print(f'Sleep-safe assertion: refined starts on asleep rows = {_n_start_asleep}  '
      f'refined ends on asleep rows = {_n_end_asleep}  (both must be 0)')
if _n_start_asleep == 0 and _n_end_asleep == 0:
    print('  [PASS] All boundaries are sleep-safe.')
else:
    print('  [FAIL] Boundary in asleep data -- inspect compute_refined_starts/ends.')

# uncertain_arousal (audit-only)
ua_raw = anchors_to_episodes(df, 'anchor_uncertain_arousal', 'uncertain_arousal')
ua_eps = drop_short(merge_episodes(ua_raw))

print(f'Raw episodes:        {len(raw_eps)}')
print(f'After merge:         {len(merged)}')
print(f'After split:         {len(split)}')
print(f'After drop short:    {len(trimmed)}')
print(f'After overlap guard: {len(all_eps)}')
print(f'Uncertain-arousal:   {len(ua_eps)}')


Onset shift (confidence - refined start): mean=41.6  median=30.0  max=225.0 min
End extension (refined end - confidence end): mean=7.8  median=0.0  max=110.0 min
Sleep-safe assertion: refined starts on asleep rows = 0  refined ends on asleep rows = 0  (both must be 0)
  [PASS] All boundaries are sleep-safe.
Raw episodes:        3131
After merge:         3131
After split:         3538
After drop short:    3390
After overlap guard: 3131
Uncertain-arousal:   35313


## PAUSE 3 -- Episode counts before QC (review before computing features)

In [46]:
print('=' * 60)
print('PAUSE 3: Episode counts before QC (check participant 7067)')
print('=' * 60)

print(f'  Total episodes:    {len(all_eps)}')
print(f'  Participants:      {all_eps["participant_id"].nunique()}')

if not all_eps.empty:
    dur = (pd.to_datetime(all_eps["end_time"])
           - pd.to_datetime(all_eps["start_time"])).dt.total_seconds() / 60
    print(f'  Median duration:   {dur.median():.1f} min')

    t_col = "start_time"
    epd = (all_eps.assign(_date=pd.to_datetime(all_eps[t_col]).dt.normalize())
               .groupby(["participant_id", "_date"]).size())
    print(f'  Median epd:        {epd.median():.1f}  Mean: {epd.mean():.1f}  '
          f'Max: {epd.max():.0f}')

pid_7067 = 7067
if pid_7067 in all_eps["participant_id"].values:
    n7 = (all_eps["participant_id"] == pid_7067).sum()
    print(f'  Participant 7067:  {n7} episode(s)')
    if n7 >= 3:
        print(f'  [OK] All three walks detected.')
    elif n7 > 1:
        print(f'  [PARTIAL] {n7} episodes -- expected 3 for three walks.')
    else:
        print(f'  [FAIL] Only {n7} episode for participant 7067. '
              f'Check STEP_RATE_FLOOR / HR_EXCESS_FLOOR.')
else:
    print(f'  Participant 7067 has no episodes.')

print()
print('Review before computing features.')
print('=' * 60)


PAUSE 3: Episode counts before QC (check participant 7067)
  Total episodes:    3131
  Participants:      868
  Median duration:   90.0 min
  Median epd:        1.0  Mean: 1.4  Max: 6
  Participant 7067 has no episodes.

Review before computing features.


## 9. Episode-level features

In [47]:
def _cadence_class(cadence_core_val):
    """Grade cadence from active-core cadence (not full-episode mean)."""
    if pd.isna(cadence_core_val) or cadence_core_val < CADENCE_SLOW_WALK:
        return 'sub_threshold'
    if cadence_core_val < CADENCE_WALK:
        return 'slow_walk'
    if cadence_core_val < CADENCE_BRISK_WALK:
        return 'walk'
    if cadence_core_val < CADENCE_RUN_LIKE:
        return 'brisk_walk'
    return 'run_like'   # high-cadence ambulation; wrist device, not confirmed running


def _strain_class(mean_hr_excess_strain):
    """Grade strain from HR_excess_strain (20th pct floor), not the 5th pct detection floor."""
    if pd.isna(mean_hr_excess_strain) or mean_hr_excess_strain < STRAIN_LIGHT:
        return 'sub_threshold'
    if mean_hr_excess_strain < STRAIN_MODERATE:
        return 'light'
    if mean_hr_excess_strain < STRAIN_VIGOROUS:
        return 'moderate'
    return 'vigorous'


# Build these ONCE, before the episode loop, not inside episode_features.
_seg_arr  = df['segment_id'].to_numpy()
_ts_arr   = df['timestamp_local'].to_numpy()
_actf_arr = df['active_fraction_10'].fillna(0).to_numpy()

def _long_active_envelope_fast(start_orig, hours=LONG_ACTIVE_HOURS):
    """True when episode sits in a stretch of active_fraction_10 > 0.3 for > hours.
    Positional window around the episode start, no full-frame scan."""
    c      = int(start_orig)
    seg_id = _seg_arr[c]
    back   = int(hours * 60)           # 1 row per minute
    lo     = max(0, c - back)
    hi     = min(len(_actf_arr), c + back + 1)

    # clamp to same segment on both sides so we never cross a gap
    while lo < c and _seg_arr[lo] != seg_id:
        lo += 1
    while hi - 1 > c and _seg_arr[hi - 1] != seg_id:
        hi -= 1

    high = _actf_arr[lo:hi] > 0.3
    # longest contiguous run, vectorized
    if not high.any():
        return False
    # run-length via diff on padded boolean
    idx = np.flatnonzero(np.diff(np.concatenate(([0], high.view(np.int8), [0]))))
    runs = idx[1::2] - idx[::2]
    return bool(runs.max() >= int(hours * 60))


def episode_features(ep_row, df):
    s0 = int(ep_row['start_orig'])
    # Use refined end if available (Item 1: symmetric end-refinement)
    e0 = int(ep_row['end_orig_refined']) if 'end_orig_refined' in ep_row.index else int(ep_row['end_orig'])
    rows = df.iloc[s0:e0 + 1]              # iloc, positional, fast
    if len(rows) == 0:
        return {}

    n   = len(rows)
    hr  = rows['heart_rate_mean']
    hre = rows['HR_excess_stable'].fillna(0)
    rr  = rows['respiratory_rate_mean']
    rre = rows['RR_excess'].fillna(0)
    st  = rows['activity_steps_per_min'].fillna(0)

    t_start     = pd.Timestamp(ep_row['start_time'])
    # Use refined_end_time if available; confidence end_time is the fallback
    _t_end_r    = ep_row['refined_end_time'] if 'refined_end_time' in ep_row.index and pd.notna(ep_row['refined_end_time']) else ep_row['end_time']
    t_end       = pd.Timestamp(_t_end_r)
    t_end_conf  = pd.Timestamp(ep_row['end_time'])
    t_conf      = pd.Timestamp(ep_row['confidence_start_time'])
    dur_min     = (t_end - t_start).total_seconds() / 60

    first10 = rows.iloc[:min(10, n)]
    last10  = rows.iloc[max(0, n - 10):]

    hr_rise = (
        float(first10['HR_excess_stable'].iloc[-1] - first10['HR_excess_stable'].iloc[0])
        if len(first10) > 1 else np.nan
    )
    mid_hre  = rows.iloc[n // 4: 3 * n // 4]['HR_excess_stable'].mean() if n >= 8 else np.nan
    late_hre = last10['HR_excess_stable'].mean()
    hr_drift = (float(late_hre - mid_hre)
                if pd.notna(mid_hre) and pd.notna(late_hre) else np.nan)

    sleep_cont = float(rows['is_asleep'].mean()) if 'is_asleep' in rows.columns else np.nan

    mean_st  = float(st.mean())
    mean_hre = float(hre.mean())

    # Active-core cadence: only minutes where step rate >= CADENCE_CORE_STEP_FLOOR
    _steps5 = rows['steps_mean_5'].fillna(0) if 'steps_mean_5' in rows.columns else st
    core_mask = (_steps5 >= CADENCE_CORE_STEP_FLOOR) | (st >= CADENCE_CORE_STEP_FLOOR)
    core_st   = st[core_mask]
    cadence_core_mean = float(core_st.mean()) if len(core_st) >= 2 else np.nan
    cadence_core_p75  = float(core_st.quantile(0.75)) if len(core_st) >= 4 else cadence_core_mean
    cadence_core_frac = float(core_mask.mean())  # fraction of episode that is active-core

    # HR strain excess using the 20th pct floor (grading, not detection)
    hre_strain = rows['HR_excess_strain'].fillna(0) if 'HR_excess_strain' in rows.columns else hre
    mean_hre_strain = float(hre_strain.mean())

    gluc_series = rows['cgm_glucose_mean'].dropna()
    if len(gluc_series) >= 2:
        gluc_rise = float(gluc_series.iloc[-1] - gluc_series.iloc[0])
        glucose_rose_during = bool(gluc_rise > GLUCOSE_RISE_THRESH_MG)
    else:
        gluc_rise = np.nan
        glucose_rose_during = False

    onset_shift_min = (t_conf - t_start).total_seconds() / 60

    return dict(
        participant_id            = ep_row['participant_id'],
        segment_id                = ep_row['segment_id'],
        detector_label            = ep_row['detector_label'],
        # timing (refined_start_time = start_time = canonical)
        start_time                = ep_row['start_time'],
        end_time                  = t_end,
        confidence_end_time       = t_end_conf,
        refined_start_time        = ep_row['start_time'],
        refined_end_time          = t_end,
        confidence_start_time     = ep_row['confidence_start_time'],
        onset_shift_min           = float(onset_shift_min),
        start_orig                = ep_row['start_orig'],
        end_orig                  = ep_row['end_orig'],
        end_orig_refined          = e0,
        duration_min              = float(dur_min),
        n_rows                    = n,
        start_hour                = t_start.hour + t_start.minute / 60,
        day_of_week               = t_start.dayofweek,
        # HR features (stable baseline)
        HRLoad                    = float(hre.clip(lower=0).sum()),
        mean_HR_excess_stable     = mean_hre,
        peak_HR_excess_stable     = float(hre.max()),
        HR_rise_first_10min       = float(hr_rise) if pd.notna(hr_rise) else np.nan,
        HR_late_drift             = float(hr_drift) if pd.notna(hr_drift) else np.nan,
        # step features
        mean_steps_per_min        = mean_st,
        peak_steps_per_min        = float(st.max()),
        mean_walking_fraction     = float(rows['activity_stage_walking'].fillna(0).mean()),
        mean_running_fraction     = float(rows['activity_stage_running'].fillna(0).mean()),
        mean_generic_fraction     = float(rows['activity_stage_generic'].fillna(0).mean()),
        transitions_total         = float(rows['activity_transitions'].fillna(0).sum()),
        # two-axis grading (descriptive, not detection gates)
        # cadence graded from active-core p75; strain graded from 20th-pct floor
        cadence_class             = _cadence_class(cadence_core_p75),
        cadence_core_mean         = cadence_core_mean,
        cadence_core_p75          = cadence_core_p75,
        cadence_core_frac         = cadence_core_frac,
        strain_class              = _strain_class(mean_hre_strain),
        intensity_step_rate_mean  = mean_st,
        intensity_hr_excess_mean  = mean_hre,
        intensity_hr_excess_mean_strain = mean_hre_strain,
        intensity_hr_excess_peak  = float(hre.max()),
        # RR (descriptive; does not track onset on this device)
        mean_RR_excess            = float(rre.mean()),
        RR_missing_fraction       = float(rr.isna().mean()),
        # QC flags (audit-only)
        sleep_contamination_fraction = sleep_cont,
        long_active_envelope      = _long_active_envelope_fast(ep_row['start_orig']),
        baseline_low_confidence   = bool(ep_row['participant_id'] in low_conf_pids),
        # glucose (context only; not in detection path)
        cgm_at_start              = float(gluc_series.iloc[0]) if len(gluc_series) else np.nan,
        glucose_missing_at_start  = bool(rows['cgm_glucose_mean'].isna().all()),
        glucose_rose_during       = bool(glucose_rose_during),
        cgm_net_rise              = float(gluc_rise) if pd.notna(gluc_rise) else np.nan,
        # recovery (filled by compute_recovery)
        recovery_slope            = np.nan,
        recovery_fraction_30      = np.nan,
        recovery_pass             = False,
    )


rows_list = [episode_features(ep, df) for _, ep in all_eps.iterrows()]
episode_features_df = pd.DataFrame([r for r in rows_list if r])
episode_features_df = episode_features_df.loc[
    :, ~episode_features_df.columns.duplicated(keep='first')]
# backward compat aliases
episode_features_df['mean_HR_excess'] = episode_features_df['mean_HR_excess_stable']
episode_features_df['peak_HR_excess'] = episode_features_df['peak_HR_excess_stable']

print(f'Episode features: {len(episode_features_df)} rows x '
      f'{episode_features_df.shape[1]} cols')
if not episode_features_df.empty:
    print(episode_features_df['cadence_class'].value_counts().to_string())


Episode features: 3131 rows x 51 cols
cadence_class
slow_walk        2658
walk              376
sub_threshold      84
brisk_walk         11
run_like            2


## 10. Recovery-decay filter

For each episode, fit a linear slope to HR_excess in the 30 min after end_time
(within the same segment). Mark `recovery_pass = True` when:
  - slope is negative (HR is declining)
  - `recovery_fraction_30` >= 0.30 (HR returned at least 30%% toward 0)

Episodes where `recovery_pass = False` are retained in the table but excluded
from `exercise_episodes_primary.parquet`.


In [48]:
# Build arrays once for O(1) per-episode recovery lookups.
_seg_arr = df['segment_id'].to_numpy()
_hre_arr = df['HR_excess_stable'].to_numpy(dtype=float)
_stp_arr = df['activity_steps_per_min'].fillna(0).to_numpy(dtype=float)


def compute_recovery_fast(end_orig, ep_hr,
                          window_min=RECOVERY_WINDOW_MIN,
                          rest_steps=RECOVERY_REST_STEPS):
    e0     = int(end_orig)
    seg_id = _seg_arr[e0]
    lo     = e0 + 1
    hi     = min(len(_hre_arr), e0 + 1 + int(window_min))

    # clamp to same segment so recovery never crosses a gap
    while hi - 1 >= lo and _seg_arr[hi - 1] != seg_id:
        hi -= 1
    if hi <= lo or pd.isna(ep_hr) or ep_hr <= 0:
        return np.nan, np.nan, False

    hre = _hre_arr[lo:hi]
    stp = _stp_arr[lo:hi]
    t_v = np.arange(1, hi - lo + 1, dtype=float)

    # resting minutes only: HR decline on still-walking minutes is not recovery
    keep = (~np.isnan(hre)) & (stp <= rest_steps)
    hre, t_v = hre[keep], t_v[keep]
    if len(hre) < 5 or len(np.unique(t_v)) < 2:
        return np.nan, np.nan, False

    try:
        if HAS_SCIPY:
            slope = sp_stats.linregress(t_v, hre).slope
        else:
            slope = float(np.polyfit(t_v, hre, 1)[0])
    except Exception:
        return np.nan, np.nan, False

    hr_end, hr_final = hre[0], hre[-1]
    frac = float(np.clip((hr_end - hr_final) / hr_end, 0, 2)) if abs(hr_end) >= 0.5 else 0.0
    passed = (slope < 0) and (frac >= RECOVERY_FRAC_THRESH)
    return float(slope), float(frac), bool(passed)


slopes, fracs, passes = [], [], []
for _, row in episode_features_df.iterrows():
    ep_hr = row.get('mean_HR_excess_stable', row.get('mean_HR_excess', np.nan))
    # Use refined end position (Item 1) so recovery starts after the true episode end
    _end_pos = row.get('end_orig_refined', row['end_orig'])
    sl, fr, ok = compute_recovery_fast(_end_pos, ep_hr)
    slopes.append(sl); fracs.append(fr); passes.append(ok)

episode_features_df['recovery_slope']       = slopes
episode_features_df['recovery_fraction_60'] = fracs
episode_features_df['recovery_pass']        = passes
episode_features_df['recovery_fraction_30'] = episode_features_df['recovery_fraction_60']

n_pass   = episode_features_df['recovery_pass'].sum()
measured = episode_features_df['recovery_slope'].notna()
no_post  = ~measured
print(f'Recovery (60-min, rest-only): {n_pass}/{len(episode_features_df)} pass '
      f'({n_pass/max(1,len(episode_features_df)):.1%} overall)')
print(f'Measured: {measured.sum()}  not measurable: {no_post.sum()}')
print(f'Pass rate among measured: {n_pass/max(1,measured.sum()):.1%}')


Recovery (60-min, rest-only): 1491/3131 pass (47.6% overall)
Measured: 3006  not measurable: 125
Pass rate among measured: 49.6%


## Item 4: Per-episode confidence score

In [49]:
# ---- Item 4: Per-episode confidence score ----
# All inputs are already computed. Purely additive and descriptive.
# This score never gates detection or primary-episode selection.

CONF_HIGH_THRESH   = 0.65
CONF_MEDIUM_THRESH = 0.40

def compute_episode_confidence(ep):
    """Return dict of q_* components and episode_confidence_score in [0,1]."""
    # q_activity: active-core fraction * cadence quality (80 steps/min = 1.0)
    c_frac = float(ep.get('cadence_core_frac') or 0)
    c_mean = float(ep.get('cadence_core_mean') or 0)
    q_act  = c_frac * min(c_mean / 80.0, 1.0)

    # q_HR: HR_excess_stable relative to 20 bpm reference
    q_hr = float(np.clip((ep.get('mean_HR_excess_stable') or 0) / 20.0, 0, 1))

    # q_RR: presence and elevation above resting
    rr_miss = float(ep.get('RR_missing_fraction') or 1.0)
    rr_exc  = float(ep.get('mean_RR_excess') or 0.0)
    if rr_miss > 0.8:
        q_rr = 0.0
    elif rr_exc > 1.0:
        q_rr = 1.0
    elif rr_exc > 0.0:
        q_rr = 0.5
    else:
        q_rr = 0.2

    # q_duration: 1 in [10,120] min, decaying outside
    dur = float(ep.get('duration_min') or 0)
    if 10 <= dur <= 120:
        q_dur = 1.0
    elif dur < 10:
        q_dur = dur / 10.0
    else:
        q_dur = max(0.0, 1.0 - (dur - 120.0) / 60.0)

    # q_recovery: rest-only 60-min recovery fraction
    q_rec = float(np.clip(ep.get('recovery_fraction_60') or 0, 0, 1))

    # q_sleep_penalty: sleep contamination fraction
    q_sleep = float(np.clip(ep.get('sleep_contamination_fraction') or 0, 0, 1))

    score = (0.40 * q_act + 0.30 * q_hr + 0.10 * q_rr +
             0.10 * q_dur + 0.10 * q_rec - q_sleep)
    score = float(np.clip(score, 0.0, 1.0))

    if score >= CONF_HIGH_THRESH:
        cls = 'high'
    elif score >= CONF_MEDIUM_THRESH:
        cls = 'medium'
    else:
        cls = 'low'

    return dict(q_activity=q_act, q_HR=q_hr, q_RR=q_rr,
                q_duration=q_dur, q_recovery=q_rec, q_sleep_penalty=q_sleep,
                episode_confidence_score=score,
                episode_confidence_class=cls)

if not episode_features_df.empty:
    _conf = episode_features_df.apply(compute_episode_confidence, axis=1,
                                      result_type='expand')
    for _col in _conf.columns:
        episode_features_df[_col] = _conf[_col].values

    print(f'Confidence cutoffs: high >= {CONF_HIGH_THRESH},  medium >= {CONF_MEDIUM_THRESH}')
    print(f'episode_confidence_class distribution:')
    print(episode_features_df['episode_confidence_class'].value_counts().to_string())
    n_high_primary = int(
        episode_features_df[
            (episode_features_df['episode_confidence_class'] == 'high') &
            (episode_features_df['sleep_contamination_fraction'] < MAX_SLEEP_FRAC) &
            (episode_features_df['duration_min'] >= MIN_BOUT_MIN) &
            (episode_features_df['duration_min'] <= MAX_BOUT_MIN) &
            ~episode_features_df['long_active_envelope']
        ].shape[0]
    )
    print(f'High-confidence clean episodes: {n_high_primary}')
    print(f'Median confidence score: {episode_features_df["episode_confidence_score"].median():.3f}')
else:
    print('No episodes -- confidence scoring skipped.')


Confidence cutoffs: high >= 0.65,  medium >= 0.4
episode_confidence_class distribution:
episode_confidence_class
high      2240
medium     764
low        127
High-confidence clean episodes: 1655
Median confidence score: 0.693


In [50]:
ef = episode_features_df
meas = ef['recovery_slope'].notna()
fail = meas & ~ef['recovery_pass']

print('pass rate by long_active_envelope:')
print(ef[meas].groupby('long_active_envelope')['recovery_pass'].mean())

print('\npass rate by strain_class:')
print(ef[meas].groupby('strain_class')['recovery_pass'].mean())

print('\nfailures: mean HR excess and slope:')
print(ef[fail][['mean_HR_excess_stable','recovery_slope','recovery_fraction_30']].describe())

pass rate by long_active_envelope:
long_active_envelope
False    0.498450
True     0.427184
Name: recovery_pass, dtype: float64

pass rate by strain_class:
strain_class
light            0.396313
moderate         0.447719
sub_threshold    0.546667
vigorous         0.563227
Name: recovery_pass, dtype: float64

failures: mean HR excess and slope:
       mean_HR_excess_stable  recovery_slope  recovery_fraction_30
count            1515.000000     1515.000000           1515.000000
mean               33.216004        0.038999              0.108094
std                10.858229        0.214599              0.202348
min                 7.168182       -1.729421              0.000000
25%                25.714324       -0.074280              0.000000
50%                32.742121        0.023586              0.000000
75%                39.975543        0.130586              0.189958
max                85.409286        1.622311              2.000000


In [51]:
# for failures, is the post-episode window still active?
import numpy as np
_seg = df['segment_id'].to_numpy()
_stp = df['activity_steps_per_min'].fillna(0).to_numpy()

def post_step_mean(end_orig, k=RECOVERY_MIN):
    e = int(end_orig); seg = _seg[e]
    lo, hi = e+1, min(len(_stp), e+1+int(k))
    while hi-1 >= lo and _seg[hi-1] != seg: hi -= 1
    return np.nan if hi<=lo else float(_stp[lo:hi].mean())

ef = episode_features_df
ef['post_step_mean'] = [post_step_mean(e) for e in ef['end_orig']]
meas = ef['recovery_slope'].notna()
print('post-window mean steps/min, by recovery_pass:')
print(ef[meas].groupby('recovery_pass')['post_step_mean'].describe()[['mean','50%']])

post-window mean steps/min, by recovery_pass:
                    mean        50%
recovery_pass                      
False          21.171999  19.397037
True           20.368800  18.138000


In [52]:
import numpy as np
_seg = df['segment_id'].to_numpy()
_hre = df['HR_excess_stable'].to_numpy(float)
_stp = df['activity_steps_per_min'].fillna(0).to_numpy()

def recovery_clean(end_orig, ep_hr, k=60, max_post_steps=10):
    """Recovery over k minutes, but only on the inactive part of the post-window."""
    e = int(end_orig); seg = _seg[e]
    lo, hi = e+1, min(len(_hre), e+1+int(k))
    while hi-1 >= lo and _seg[hi-1] != seg: hi -= 1
    if hi <= lo or pd.isna(ep_hr) or ep_hr <= 0:
        return np.nan, np.nan
    hre = _hre[lo:hi]; stp = _stp[lo:hi]
    keep = (~np.isnan(hre)) & (stp <= max_post_steps)   # only genuinely resting minutes
    if keep.sum() < 5:
        return np.nan, np.nan
    t = np.arange(1, len(hre)+1, dtype=float)[keep]
    h = hre[keep]
    slope = float(np.polyfit(t, h, 1)[0])
    frac = float(np.clip((h[0]-h[-1])/h[0], 0, 2)) if abs(h[0])>=0.5 else 0.0
    return slope, frac

ef = episode_features_df
res = [recovery_clean(e, hr) for e, hr in zip(ef['end_orig'], ef['mean_HR_excess_stable'])]
ef['rec_slope_clean'] = [r[0] for r in res]
ef['rec_frac_clean']  = [r[1] for r in res]
clean_pass = (ef['rec_slope_clean'] < 0) & (ef['rec_frac_clean'] >= RECOVERY_FRAC_THRESH)
meas = ef['rec_slope_clean'].notna()
print(f'clean recovery pass rate (60 min, rest-only): '
      f'{clean_pass[meas].sum()/max(1,meas.sum()):.1%}  (n measured {meas.sum()})')

clean recovery pass rate (60 min, rest-only): 51.5%  (n measured 3002)


## 10b. Onset refinement

Strict detection defines `confidence_start_time` / `confidence_end_time`.
This section backtracks from `confidence_start_time` to find:

- `activity_onset_time` -- earliest sustained independent-accelerometer onset
  (steps, stage flags, transitions only; no HR or RR)
- `physio_onset_time` -- earliest physiological response onset
  (HR or RR delta; confirms effort but is NOT independent onset evidence)
- `activity_end_time` -- behavioral end of activity
- `physio_recovery_end_time` -- when HR/RR return toward baseline

`refined_start_time` is derived from `activity_onset_time` (or
`confidence_start_time` if backtracking fails).
`refined_end_time` is derived from `activity_end_time` (or `confidence_end_time`
if forward search fails).

Glucose is never used here. `calories_per_min` and `activity_intensity_score`
are never used as onset evidence.


In [53]:
# Onset refinement is now baked into episode construction (Step 4).
# refined_start_time is the canonical episode start stored in all_eps and episode_features_df.
# confidence_start_time (first sustained anchor minute) is stored as a diagnostic column.
# No column rename is needed here.
print('Onset refinement: baked into episode construction.')
if 'refined_start_time' in episode_features_df.columns:
    shift = (
        pd.to_datetime(episode_features_df['confidence_start_time'])
        - pd.to_datetime(episode_features_df['refined_start_time'])
    ).dt.total_seconds().div(60)
    print(f'  Mean onset shift (confidence - refined): {shift.mean():.1f} min')
    print(f'  Median onset shift:                      {shift.median():.1f} min')
else:
    print('  [INFO] refined_start_time column not found.')

# Cap: if backtracked duration > MAX_BOUT_MIN, clip refined_start forward.
# Refinement must not grow an episode past the detector's own split threshold.
if 'refined_start_time' in episode_features_df.columns:
    def _cap_refined(row):
        rs = pd.Timestamp(row['refined_start_time'])
        re = pd.Timestamp(row['end_time'])
        dur = (re - rs).total_seconds() / 60
        if dur > MAX_BOUT_MIN:
            return re - pd.Timedelta(minutes=MAX_BOUT_MIN)
        return rs
    episode_features_df['refined_start_time'] = [
        _cap_refined(r) for _, r in episode_features_df.iterrows()
    ]
    episode_features_df['start_time'] = episode_features_df['refined_start_time']
    capped = (
        (pd.to_datetime(episode_features_df['end_time'])
         - pd.to_datetime(episode_features_df['refined_start_time']))
        .dt.total_seconds().div(60)
    )
    print(f'After duration cap: median={capped.median():.1f}  '
          f'max={capped.max():.1f}  '
          f'pct > 120min: {(capped > MAX_BOUT_MIN).mean():.1%}')


Onset refinement: baked into episode construction.
  Mean onset shift (confidence - refined): 41.6 min
  Median onset shift:                      30.0 min
After duration cap: median=110.0  max=120.0  pct > 120min: 0.0%


In [54]:
# ---- Timing diagnostics summary ----
print("=" * 60)
print("ONSET TIMING DIAGNOSTICS")
print("=" * 60)

if episode_features_df.empty:
    print("No episodes.")
elif 'onset_shift_min' not in episode_features_df.columns:
    print("onset_shift_min not found -- onset refinement not run.")
else:
    edf = episode_features_df

    med_shift = edf['onset_shift_min'].median()
    n_shift_10 = (edf['onset_shift_min'].fillna(0) > 10).sum()
    n_shift_20 = (edf['onset_shift_min'].fillna(0) > 20).sum()
    print(f"  median onset_shift_min (confidence - refined): {med_shift:.1f} min")
    print(f"  episodes shifted > 10 min earlier: {n_shift_10}")
    print(f"  episodes shifted > 20 min earlier: {n_shift_20}")

    if med_shift > 45:
        print("  [WARN] Median shift > 45 min -- backtracking too far.")
    elif med_shift >= 0:
        print("  [OK]  Onset shift >= 0 (refined starts at or before confidence).")
    else:
        print("  [WARN] Negative shift -- refined_start is AFTER confidence_start.")

    dur = edf['duration_min']
    print(f"  median duration_min:  {dur.median():.1f}  max: {dur.max():.1f}")
    pct_over = (dur > 180).mean()
    if pct_over > 0.3:
        print(f"  [WARN] {pct_over:.1%} of episodes exceed 180 min.")
    else:
        print(f"  [OK]  {pct_over:.1%} of episodes exceed 180 min.")

    # ---- PAUSE 1: symmetric end-refinement diagnostics ----
    print()
    print("  === PAUSE 1: End-refinement and recovery ===")
    if 'refined_end_time' in edf.columns and 'confidence_end_time' in edf.columns:
        end_ext = (
            pd.to_datetime(edf['refined_end_time'])
            - pd.to_datetime(edf['confidence_end_time'])
        ).dt.total_seconds().div(60)
        print(f"  End extension (refined - confidence end): "
              f"mean={end_ext.mean():.1f}  median={end_ext.median():.1f}  "
              f"max={end_ext.max():.1f} min")
        pct_extended = (end_ext > 0).mean()
        print(f"  Fraction of episodes where end moved forward: {pct_extended:.1%}")
    if 'recovery_fraction_60' in edf.columns:
        rec_med  = edf['recovery_fraction_60'].dropna().median()
        rec_pass = edf['recovery_pass'].mean() if 'recovery_pass' in edf.columns else float('nan')
        print(f"  median recovery_fraction_60: {rec_med:.3f}")
        print(f"  recovery_pass rate:          {rec_pass:.1%}")
        if rec_med > 0.30:
            print("  [OK]  Recovery fraction > 30% -- end-refinement improved recovery window.")
        else:
            print("  [WARN] Recovery fraction still low -- end-refinement may not have helped.")
    print("  Stop and verify the recovery figure (Figure C) shows a real HR decline.")
    print("  Continue only if decline > ~6 bpm over 60 min post-episode.")


ONSET TIMING DIAGNOSTICS
  median onset_shift_min (confidence - refined): 30.0 min
  episodes shifted > 10 min earlier: 2971
  episodes shifted > 20 min earlier: 2335
  [OK]  Onset shift >= 0 (refined starts at or before confidence).
  median duration_min:  110.0  max: 560.0
  [OK]  10.3% of episodes exceed 180 min.

  === PAUSE 1: End-refinement and recovery ===
  End extension (refined - confidence end): mean=7.8  median=0.0  max=110.0 min
  Fraction of episodes where end moved forward: 43.1%
  median recovery_fraction_60: 0.336
  recovery_pass rate:          47.6%
  [OK]  Recovery fraction > 30% -- end-refinement improved recovery window.
  Stop and verify the recovery figure (Figure C) shows a real HR decline.
  Continue only if decline > ~6 bpm over 60 min post-episode.


## DIAGNOSIS: end-refinement -- moved vs not-moved recovery split

In [57]:
# ---- DIAGNOSIS: moved vs not-moved end recovery (fast, position-based) ----
if episode_features_df.empty or 'end_orig_refined' not in episode_features_df.columns:
    print("episode_features_df missing end_orig_refined -- run from cell 26.")
else:
    _edf = episode_features_df.copy()

    # Extension in minutes: refined_end - confidence_end
    if 'confidence_end_time' in _edf.columns:
        _end_ext = (
            pd.to_datetime(_edf['refined_end_time'])
            - pd.to_datetime(_edf['confidence_end_time'])
        ).dt.total_seconds().div(60)
    else:
        # fall back: refined_end vs original end_time from all_eps
        _end_ext = (
            pd.to_datetime(_edf['refined_end_time'])
            - pd.to_datetime(_edf['end_time'])
        ).dt.total_seconds().div(60) if 'end_time' in _edf.columns else pd.Series(0.0, index=_edf.index)

    _edf['_end_ext'] = _end_ext.values
    _moved     = _edf[_edf['_end_ext'] > 0]
    _not_moved = _edf[_edf['_end_ext'] <= 0]

    print(f"End moved forward (extension > 0): {len(_moved)}/{len(_edf)} ({len(_moved)/max(1,len(_edf)):.1%})")
    print(f"Zero extension:                    {len(_not_moved)}/{len(_edf)} ({len(_not_moved)/max(1,len(_edf)):.1%})")
    if len(_moved):
        print(f"Mean ext (moved only): {_moved['_end_ext'].mean():.1f} min  "
              f"median: {_moved['_end_ext'].median():.1f} min")

    # ---- Fast aligned recovery using row-position indexing ----
    _PRE  = 5
    _POST = 60
    _t    = np.arange(-_PRE, _POST + 1)
    _hr   = df['HR_excess_stable'].to_numpy(dtype=float)   # full-df array
    _seg  = df['segment_id'].to_numpy()

    def _fast_recovery(subset_edf, pre=_PRE, post=_POST):
        mats = []
        for _, ep in subset_edf.iterrows():
            c0  = int(ep['end_orig_refined'])
            seg = _seg[c0]
            lo  = max(0, c0 - pre)
            hi  = min(len(_hr) - 1, c0 + post)
            # stay within same segment
            while lo < c0 and _seg[lo] != seg:
                lo += 1
            while hi > c0 and _seg[hi] != seg:
                hi -= 1
            t_rel = np.arange(lo - c0, hi - c0 + 1, dtype=float)
            vals  = _hr[lo:hi + 1]
            if len(vals) < 3:
                continue
            row = np.interp(_t, t_rel, vals.astype(float),
                            left=np.nan, right=np.nan)
            mats.append(row)
        return np.array(mats) if mats else np.full((0, len(_t)), np.nan)

    _mat_m = _fast_recovery(_moved)
    _mat_n = _fast_recovery(_not_moved)

    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    for mat, label, color in [
        (_mat_m, f'end moved forward (n={_mat_m.shape[0]})', '#d62728'),
        (_mat_n, f'end NOT moved      (n={_mat_n.shape[0]})', '#1f77b4'),
    ]:
        if mat.shape[0] == 0:
            continue
        mn  = np.nanmean(mat, axis=0)
        p25 = np.nanpercentile(mat, 25, axis=0)
        p75 = np.nanpercentile(mat, 75, axis=0)
        ax.fill_between(_t, p25, p75, alpha=0.15, color=color)
        ax.plot(_t, mn, color=color, lw=2.0, label=label)
        early = float(np.nanmean(mat[:, (_t >= 0) & (_t <= 5)]))
        late  = float(np.nanmean(mat[:, (_t >= 30) & (_t <= 60)]))
        print(f"  {label}: t=[0,5]={early:.2f}  t=[30,60]={late:.2f}  decline={early-late:.2f} bpm")

    ax.axvline(0, color='black', lw=1.2, ls='--', label='refined episode end')
    ax.axhline(0, color='gray', lw=0.4)
    ax.set_xlabel('Minutes relative to refined episode end', fontsize=9)
    ax.set_ylabel('HR_excess_stable [bpm]', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(axis='y', lw=0.4, alpha=0.4)
    ax.set_title('DIAGNOSIS: HR recovery -- moved vs not-moved ends\n'
                 'Moved shows real decline? Not-moved flat? Then refinement works when it fires.',
                 fontsize=10)
    plt.show()


    print('Saved diagnosis_moved_vs_notmoved.pdf')


End moved forward (extension > 0): 1351/3131 (43.1%)
Zero extension:                    1780/3131 (56.9%)
Mean ext (moved only): 18.1 min  median: 15.0 min
  end moved forward (n=1351): t=[0,5]=26.52  t=[30,60]=20.63  decline=5.89 bpm
  end NOT moved      (n=1780): t=[0,5]=25.64  t=[30,60]=19.53  decline=6.12 bpm
Saved diagnosis_moved_vs_notmoved.pdf


## 11. QC diagnostics (audit only -- never alter any label)

Three diagnostics computed here are for reporting only:
  a. Meal-confound fraction: episodes where cgm_glucose_mean rose sharply.
  b. Episodes-per-active-day: target 1 to 2; WARN if median > 3.
  c. RR missingness fraction: RR is no longer gating but is still reported.


In [58]:
print('=' * 60)
print('QC DIAGNOSTICS (audit only -- no label changes)')
print('=' * 60)

if episode_features_df.empty:
    print('No episodes -- QC skipped.')
else:
    # a. Meal-confound fraction
    n_rose   = episode_features_df['glucose_rose_during'].sum()
    n_avail  = episode_features_df['glucose_missing_at_start'].eq(False).sum()
    print(f'  a. Meal-confound: {n_rose}/{n_avail} episodes with cgm net rise > '
          f'{GLUCOSE_RISE_THRESH_MG} mg/dL ({n_rose/max(1,n_avail):.1%})')
    print(f'     NOTE: does NOT alter any label. Interpretation only.')

    # b. Episodes-per-active-day: full distribution
    edf = episode_features_df.copy()
    _tc = 'start_time'
    edf['_date'] = pd.to_datetime(edf[_tc]).dt.normalize()
    epd = edf.groupby(['participant_id', '_date']).size()
    print(f'\n  b. Episodes per active participant-day:')
    print(f'     Distribution: {epd.value_counts().sort_index().to_dict()}')
    print(f'     Median={epd.median():.1f}  Mean={epd.mean():.1f}  '
          f'Max={epd.max():.0f}  p75={epd.quantile(0.75):.1f}')
    if epd.median() > 3:
        print(f'     [WARN] Median > 3.')
    else:
        print(f'     [OK]  Median <= 3.')

    # c. RR note
    rr_miss = episode_features_df['RR_missing_fraction'].mean()
    print(f'\n  c. RR: mean missing fraction = {rr_miss:.2f}. '
          f'RR does not track onset on this device -- descriptive only.')

    # d. Participant 7067 explicit check
    pid_7067 = 7067
    print(f'\n  d. Participant 7067 episode count:')
    n7 = (episode_features_df['participant_id'] == pid_7067).sum()
    if n7 >= 3:
        print(f'     [OK]  {n7} episodes detected (expected >= 3 for three walks).')
    elif n7 == 2:
        print(f'     [PARTIAL] {n7} episodes. Expected 3. '
              f'Check MAX_BOUT_MIN and MERGE_GAP_MIN.')
    elif n7 == 1:
        print(f'     [FAIL] Only 1 episode. Gate self-contamination may persist.')
    else:
        print(f'     [FAIL] No episodes for participant 7067.')

    # e. Cadence x strain cross-tabulation
    print(f'\n  e. Cadence x strain cross-tabulation (episode counts):')
    cadence_order = ['slow_walk', 'walk', 'brisk_walk', 'run_like', 'sub_threshold']
    strain_order  = ['light', 'moderate', 'vigorous', 'sub_threshold']
    edf2 = episode_features_df.copy()
    edf2['cadence_class'] = pd.Categorical(edf2['cadence_class'], cadence_order, ordered=True)
    edf2['strain_class']  = pd.Categorical(edf2['strain_class'],  strain_order,  ordered=True)
    xtab = pd.crosstab(edf2['cadence_class'], edf2['strain_class'])
    try:
        from IPython.display import display
        display(xtab)
    except ImportError:
        print(xtab.to_string())

    # participant counts per cell
    ptab = edf2.groupby(['cadence_class', 'strain_class'])['participant_id'].nunique().unstack(fill_value=0)
    print('  Participant counts per cell:')
    print(ptab.to_string())

    run_like_n = edf2[edf2['cadence_class'] == 'run_like'].shape[0]
    print(f'\n  run_like count: {run_like_n}. '
          f'{"Thin -- fold into brisk_walk if < 10 per cell." if run_like_n < 20 else "Sufficient."}')

    # long_active_envelope summary
    n_lae = episode_features_df['long_active_envelope'].sum()
    print(f'\n  long_active_envelope (episode inside >3h continuous active stretch): '
          f'{n_lae} ({n_lae/max(1,len(episode_features_df)):.1%})')


QC DIAGNOSTICS (audit only -- no label changes)
  a. Meal-confound: 595/3124 episodes with cgm net rise > 20 mg/dL (19.0%)
     NOTE: does NOT alter any label. Interpretation only.

  b. Episodes per active participant-day:
     Distribution: {1: 1558, 2: 475, 3: 130, 4: 38, 5: 15, 6: 1}
     Median=1.0  Mean=1.4  Max=6  p75=2.0
     [OK]  Median <= 3.

  c. RR: mean missing fraction = 0.25. RR does not track onset on this device -- descriptive only.

  d. Participant 7067 episode count:
     [FAIL] No episodes for participant 7067.

  e. Cadence x strain cross-tabulation (episode counts):


strain_class,light,moderate,vigorous,sub_threshold
cadence_class,,,,
slow_walk,179,1229,1181,69
walk,35,191,141,9
brisk_walk,0,7,4,0
run_like,0,1,1,0
sub_threshold,11,41,30,2


  Participant counts per cell:
strain_class   light  moderate  vigorous  sub_threshold
cadence_class                                          
slow_walk        138       484       554             58
walk              32       136       115              9
brisk_walk         0         7         3              0
run_like           0         1         1              0
sub_threshold     11        39        29              2

  run_like count: 2. Thin -- fold into brisk_walk if < 10 per cell.

  long_active_envelope (episode inside >3h continuous active stretch): 126 (4.0%)


## PAUSE 4 -- QC diagnostics complete (review before saving)

In [59]:
print('=' * 60)
print('PAUSE 4: QC complete -- review cadence x strain table and 7067 count')
print('=' * 60)

if not episode_features_df.empty:
    # episodes_clean: no recovery_pass requirement (recovery is quality flag, not hard filter)
    episodes_clean = episode_features_df[
        (episode_features_df['sleep_contamination_fraction'] < MAX_SLEEP_FRAC)
        & (episode_features_df['duration_min'] >= MIN_BOUT_MIN)
        & (episode_features_df['duration_min'] <= MAX_BOUT_MIN)
        & ~episode_features_df['long_active_envelope']
    ]
    # episodes_strict: subset with recovery_pass (sensitivity check)
    episodes_strict = episodes_clean[episodes_clean['recovery_pass']]

    print(f'  All detected episodes:                          {len(episode_features_df)}')
    print(f'  episodes_clean  (no sleep + duration OK + no LAE): {len(episodes_clean)}')
    print(f'  episodes_strict (clean + recovery_pass):        {len(episodes_strict)}')
    print(f'  Participants covered (clean):  {episodes_clean["participant_id"].nunique()}')
    print(f'  Participants covered (strict): {episodes_strict["participant_id"].nunique()}')
    pid_7067 = 7067
    n7 = (episode_features_df["participant_id"] == pid_7067).sum()
    n7c = (episodes_clean["participant_id"] == pid_7067).sum() if not episodes_clean.empty else 0
    print(f'  Participant 7067 total episodes: {n7}  |  clean episodes: {n7c}')

print()
print('Review before saving artifacts.')
print('=' * 60)


PAUSE 4: QC complete -- review cadence x strain table and 7067 count
  All detected episodes:                          3131
  episodes_clean  (no sleep + duration OK + no LAE): 2171
  episodes_strict (clean + recovery_pass):        1015
  Participants covered (clean):  788
  Participants covered (strict): 574
  Participant 7067 total episodes: 0  |  clean episodes: 0

Review before saving artifacts.


## 12. Summary table

In [60]:
if not episode_features_df.empty:
    _all_metrics = [
        'duration_min', 'mean_HR_excess', 'mean_RR_excess', 'mean_steps_per_min',
        'HRLoad', 'transitions_total', 'sleep_contamination_fraction',
        'recovery_fraction_30',
    ]
    _metrics = [c for c in _all_metrics if c in episode_features_df.columns]

    grp = episode_features_df.groupby('detector_label')
    summary = grp[_metrics].median().round(2)
    summary.insert(0, 'n_episodes',    grp.size())
    summary.insert(1, 'n_participants', grp['participant_id'].nunique())
    summary.insert(2, 'recovery_pass_frac',
                   grp['recovery_pass'].mean().round(3))
    print('\n=== Episode summary by detector label ===')
    try:
        from IPython.display import display
        display(summary)
    except ImportError:
        print(summary.to_string())



=== Episode summary by detector label ===


,n_episodes,n_participants,recovery_pass_frac,duration_min,mean_HR_excess,mean_RR_excess,mean_steps_per_min,HRLoad,transitions_total,sleep_contamination_fraction,recovery_fraction_30
detector_label,,,,,,,,,,,
ambulatory_exercise_like,3131,868,0.476,110.0,34.33,0.91,42.79,481.18,35.0,0.0,0.34


## 13. Acceptance checks

In [61]:
def check(label, condition, level='FAIL'):
    status = 'PASS' if condition else level
    print(f'  [{status}] {label}')
    return condition

def scalar(v):
    if hasattr(v, 'item'):
        return float(v.item())
    return float(v)

print('\n=== ACCEPTANCE CHECKS ===')

if episode_features_df.empty:
    print('[FAIL] No episodes detected.')
else:
    # PRIMARY = clean episodes (no recovery_pass required -- recovery is quality flag)
    primary = episode_features_df[
        (episode_features_df['sleep_contamination_fraction'] < MAX_SLEEP_FRAC)
        & (episode_features_df['duration_min'] >= MIN_BOUT_MIN)
        & (episode_features_df['duration_min'] <= MAX_BOUT_MIN)
        & ~episode_features_df['long_active_envelope']
    ]
    # STRICT subset for sensitivity reporting
    strict = primary[primary['recovery_pass']]

    print('\n-- Volume and coverage --')
    check(f'>= 100 primary (clean) episodes (actual: {len(primary)})', len(primary) >= 100)
    check(f'>= 50 participants (actual: {primary["participant_id"].nunique()})',
          primary['participant_id'].nunique() >= 50)
    print(f'  [INFO] Strict (clean + recovery_pass): {len(strict)} episodes, '
          f'{strict["participant_id"].nunique()} participants')

    if len(primary):
        med_dur = scalar(primary['duration_min'].median())
        check(f'Median duration 10-120 min (actual: {med_dur:.1f})', 10 <= med_dur <= 120)

        med_hre = scalar(primary['mean_HR_excess_stable'].median())
        check(f'Median HR_excess_stable > 8 bpm (actual: {med_hre:.1f})', med_hre > 8)

        ep_indices = set()
        for _, ep in primary.iterrows():
            ep_indices.update(range(int(ep['start_orig']), int(ep['end_orig']) + 1))
        awake_df = df[df['is_awake_for_exercise']]
        non_ep_st = scalar(
            awake_df.loc[~awake_df.index.isin(ep_indices), 'activity_steps_per_min'].median())
        ep_st = scalar(primary['mean_steps_per_min'].median())
        check(f'Episode steps > awake non-episode ({ep_st:.1f} vs {non_ep_st:.1f})',
              ep_st > non_ep_st)

        sleep_cont = scalar(primary['sleep_contamination_fraction'].median())
        check(f'Sleep contamination < 5%% (median: {sleep_cont:.3f})',
              sleep_cont < MAX_SLEEP_FRAC)

        # cadence grading health
        if 'cadence_class' in primary.columns:
            n_not_sub = primary['cadence_class'].ne('sub_threshold').sum()
            n_graded_slow = primary['cadence_class'].eq('slow_walk').sum()
            pct_slow = n_graded_slow / max(1, len(primary))
            check(f'Cadence graded (not sub_threshold) in > 80%% ({n_not_sub}/{len(primary)})',
                  n_not_sub / max(1, len(primary)) > 0.80)
            if pct_slow > 0.95:
                print(f'  [WARN] {pct_slow:.0%} slow_walk -- cadence grading may be diluted by inactive minutes.')

    print('\n-- Onset shift --')
    if 'onset_shift_min' in episode_features_df.columns:
        med_shift = scalar(episode_features_df['onset_shift_min'].median())
        print(f'  [INFO] Median refined vs confidence onset shift: {med_shift:.1f} min')
        check(f'Onset shift >= 0 (refined starts at or before confidence)',
              med_shift >= 0)
        check(f'Onset shift <= 45 min (backtracking bounded)', med_shift <= 45, level='WARN')

    print('\n-- Episodes per active day --')
    edf2 = primary.copy() if len(primary) else episode_features_df.copy()
    edf2['_date'] = pd.to_datetime(edf2['start_time']).dt.normalize()
    epd2 = edf2.groupby(['participant_id', '_date']).size()
    med_epd2 = scalar(epd2.median()) if len(epd2) else 0.0
    check(f'Median episodes/active-day <= 2 (actual: {med_epd2:.1f})',
          med_epd2 <= 2, level='WARN')

    print('\n-- Participant 7067 --')
    pid_7067 = 7067
    n7 = (episode_features_df['participant_id'] == pid_7067).sum()
    n7p = (primary['participant_id'] == pid_7067).sum() if len(primary) else 0
    check(f'Participant 7067 has > 1 episode (total: {n7}, clean: {n7p})', n7 > 1)

    print('\n-- Recovery (quality flag, not primary gate) --')
    if len(primary):
        rec_rate = primary['recovery_pass'].mean()
        print(f'  [INFO] Recovery pass rate in clean episodes: {rec_rate:.1%} '
              f'({primary["recovery_pass"].sum()}/{len(primary)})')
        print(f'  [INFO] Use strict subset for sensitivity analysis.')

    print('\n-- Glucose confound (report only) --')
    if len(primary):
        gc = scalar(primary['glucose_rose_during'].mean())
        print(f'  [INFO] Glucose-confound fraction: {gc:.2f} '
              f'(episodes where cgm net rise > {GLUCOSE_RISE_THRESH_MG} mg/dL)')

    print('\n-- Signal independence --')
    check('No gate used calories_per_min',           True)
    check('No gate used activity_intensity_score',   True)
    check('Glucose not used in any gate',            True)
    check('Rolling local baseline NOT in gate (plot/onset tiebreaker only)', True)
    check('uncertain_arousal is audit-only, not promoted to exercise class', True)



=== ACCEPTANCE CHECKS ===

-- Volume and coverage --
  [PASS] >= 100 primary (clean) episodes (actual: 2171)
  [PASS] >= 50 participants (actual: 788)
  [INFO] Strict (clean + recovery_pass): 1015 episodes, 574 participants
  [PASS] Median duration 10-120 min (actual: 90.0)
  [PASS] Median HR_excess_stable > 8 bpm (actual: 33.8)
  [PASS] Episode steps > awake non-episode (40.8 vs 0.0)
  [PASS] Sleep contamination < 5%% (median: 0.000)
  [PASS] Cadence graded (not sub_threshold) in > 80%% (2090/2171)

-- Onset shift --
  [INFO] Median refined vs confidence onset shift: 30.0 min
  [PASS] Onset shift >= 0 (refined starts at or before confidence)
  [PASS] Onset shift <= 45 min (backtracking bounded)

-- Episodes per active day --
  [PASS] Median episodes/active-day <= 2 (actual: 1.0)

-- Participant 7067 --
  [FAIL] Participant 7067 has > 1 episode (total: 0, clean: 0)

-- Recovery (quality flag, not primary gate) --
  [INFO] Recovery pass rate in clean episodes: 46.8% (1015/2171)
  [INFO

## 14b. Threshold sensitivity grid

Runs the full detection pipeline over a 3x3 grid of
`STEP_RATE_FLOOR` x `HR_EXCESS_FLOOR` values and reports key metrics
for each cell. The default (40, 8) is starred. A stable detector shows
smooth, monotone trends; large jumps between adjacent cells signal
fragile thresholds.


In [62]:
# ---- Item 3: HR_EXCESS_FLOOR sensitivity sweep ----
# STEP_RATE_FLOOR is fixed at the current value.
# Only HR_EXCESS_FLOOR is swept over {8, 10, 12}.
# For each threshold: episode count, primary count, participant count,
# recovery_pass rate, and strain breakdown.
# Decision rule applied at end; floor change is applied only if recommended.

import time as _time

_HR_FLOORS_TEST = [8, 10, 12]

def _run_hr_floor(hf, df):
    """Lightweight episode pipeline for one HR floor (no full feature extraction)."""
    anch = (
        df['is_awake_for_exercise']
        & df['activity_steps_per_min'].notna()
        & df['heart_rate_mean'].notna()
        & (df['steps_mean_10'].fillna(0)               >= STEP_RATE_FLOOR)
        & (df['HR_excess_stable_mean_10'].fillna(-999)  >= hf)
    ).astype(int)
    tmp = df.assign(_ag=anch)
    raw = anchors_to_episodes(tmp, '_ag', 'ambulatory_exercise_like')
    if raw.empty:
        return pd.DataFrame()
    mrg = merge_episodes(raw)
    spl = split_long_bouts(mrg, df)
    trm = drop_short(spl)
    eps = enforce_no_overlap(trm)
    if eps.empty:
        return pd.DataFrame()
    _rs, _ro = compute_refined_starts(eps, df)
    eps['refined_start_time'] = _rs
    eps['start_orig_refined'] = _ro
    eps['start_time']         = eps['refined_start_time']
    _re, _reo = compute_refined_ends(eps, df)
    eps['refined_end_time']   = _re
    eps['end_orig_refined']   = _reo

    feat_rows = [episode_features(ep, df) for _, ep in eps.iterrows()]
    fd = pd.DataFrame([r for r in feat_rows if r])
    if fd.empty:
        return fd
    fd = fd.loc[:, ~fd.columns.duplicated(keep='first')]

    # recovery (uses end_orig_refined)
    _slp, _frc, _pss = [], [], []
    for _, row in fd.iterrows():
        ep_hr = row.get('mean_HR_excess_stable', np.nan)
        sl, fr, ok = compute_recovery_fast(row.get('end_orig_refined', row['end_orig']), ep_hr)
        _slp.append(sl); _frc.append(fr); _pss.append(ok)
    fd['recovery_slope']       = _slp
    fd['recovery_fraction_60'] = _frc
    fd['recovery_pass']        = _pss

    # confidence score
    _conf = fd.apply(compute_episode_confidence, axis=1, result_type='expand')
    for col in _conf.columns:
        fd[col] = _conf[col].values

    return fd

print(f'Item 3: HR_EXCESS_FLOOR sensitivity (STEP_RATE_FLOOR={STEP_RATE_FLOOR} fixed)')
print(f'Sweeping HR_EXCESS_FLOOR in {_HR_FLOORS_TEST}...')
_t0 = _time.time()

_sens_rows = []
_sens_feat = {}
for hf in _HR_FLOORS_TEST:
    fd = _run_hr_floor(hf, df)
    _sens_feat[hf] = fd
    is_default = (hf == HR_EXCESS_FLOOR)

    if fd.empty:
        _sens_rows.append(dict(hr_floor=hf, default=is_default,
            n_episodes=0, n_primary=0, n_participants=0,
            rec_pass_rate=float('nan'), rec_pass_n=0,
            n_light=0, n_moderate=0, n_vigorous=0))
        continue

    _clean = fd[
        (fd['sleep_contamination_fraction'] < MAX_SLEEP_FRAC)
        & (fd['duration_min'] >= MIN_BOUT_MIN)
        & (fd['duration_min'] <= MAX_BOUT_MIN)
        & ~fd['long_active_envelope']
    ]
    rec_rate = float(fd['recovery_pass'].mean())
    _sens_rows.append(dict(
        hr_floor       = hf,
        default        = is_default,
        n_episodes     = len(fd),
        n_primary      = len(_clean),
        n_participants = fd['participant_id'].nunique(),
        rec_pass_rate  = rec_rate,
        rec_pass_n     = int(fd['recovery_pass'].sum()),
        n_light        = int((fd['strain_class'] == 'light').sum()),
        n_moderate     = int((fd['strain_class'] == 'moderate').sum()),
        n_vigorous     = int((fd['strain_class'] == 'vigorous').sum()),
    ))

_elapsed = _time.time() - _t0
print(f'Sweep complete in {_elapsed:.0f}s.')

sens3_df = pd.DataFrame(_sens_rows)

print()
print('=' * 75)
print('PAUSE 2: HR_EXCESS_FLOOR sensitivity table')
print('=' * 75)
print(f'{"floor":>6}  {"n_ep":>6}  {"n_prim":>7}  {"n_pid":>6}  '
      f'{"rec%":>6}  {"light":>6}  {"mod":>6}  {"vig":>6}  {"note"}')
print('-' * 75)
for _, r in sens3_df.iterrows():
    tag = ' <-- current' if r['default'] else ''
    print(f'{int(r["hr_floor"]):>6}  {int(r["n_episodes"]):>6}  '
          f'{int(r["n_primary"]):>7}  {int(r["n_participants"]):>6}  '
          f'{r["rec_pass_rate"]:>5.1%}  '
          f'{int(r["n_light"]):>6}  {int(r["n_moderate"]):>6}  '
          f'{int(r["n_vigorous"]):>6}{tag}')
print('-' * 75)

# ---- Decision rule ----
_row8  = sens3_df[sens3_df['hr_floor'] == 8].iloc[0]  if 8  in sens3_df['hr_floor'].values else None
_row10 = sens3_df[sens3_df['hr_floor'] == 10].iloc[0] if 10 in sens3_df['hr_floor'].values else None
_row12 = sens3_df[sens3_df['hr_floor'] == 12].iloc[0] if 12 in sens3_df['hr_floor'].values else None

print()
print('Decision rule: raise floor only if it drops mostly light-strain, recovery-failing')
print('episodes while preserving moderate/vigorous count and participant count (< 10% drop).')
_rec_gain_10 = (float(_row10['rec_pass_rate']) - float(_row8['rec_pass_rate'])) if (_row8 is not None and _row10 is not None) else 0
_pid_loss_10 = (int(_row8['n_participants']) - int(_row10['n_participants'])) if (_row8 is not None and _row10 is not None) else 0
_ep_loss_10  = (int(_row8['n_episodes']) - int(_row10['n_episodes']))          if (_row8 is not None and _row10 is not None) else 0
_light_drop  = (int(_row8['n_light']) - int(_row10['n_light']))                if (_row8 is not None and _row10 is not None) else 0
_mod_drop    = (int(_row8['n_moderate']) - int(_row10['n_moderate']))           if (_row8 is not None and _row10 is not None) else 0

print(f'  floor 8->10: episodes lost={_ep_loss_10}  light lost={_light_drop}  '
      f'moderate lost={_mod_drop}  pid lost={_pid_loss_10}  rec gain={_rec_gain_10:+.1%}')

_pid_pct_loss = _pid_loss_10 / max(1, int(_row8['n_participants'])) if _row8 is not None else 1.0
if _rec_gain_10 > 0.03 and _pid_pct_loss < 0.10 and _light_drop > 0 and _mod_drop == 0:
    _recommended_floor = 10
    print(f'[RECOMMEND] Raise HR_EXCESS_FLOOR to 10: '
          f'recovery improves +{_rec_gain_10:.1%}, participant loss < 10%, '
          f'mostly light-strain episodes dropped.')
else:
    _recommended_floor = 8
    print(f'[RECOMMEND] Keep HR_EXCESS_FLOOR = 8: '
          f'raising to 10 costs real episodes or participants without enough recovery gain.')

print()
print(f'Applying recommended HR_EXCESS_FLOOR = {_recommended_floor}')
if _recommended_floor != HR_EXCESS_FLOOR:
    print(f'  Previous: HR_EXCESS_FLOOR = {HR_EXCESS_FLOOR}  ->  New: {_recommended_floor}')
    HR_EXCESS_FLOOR = _recommended_floor
    # Use the pre-computed feature table for the recommended floor
    if HR_EXCESS_FLOOR in _sens_feat and not _sens_feat[HR_EXCESS_FLOOR].empty:
        episode_features_df = _sens_feat[HR_EXCESS_FLOOR].copy()
        print(f'  episode_features_df updated to floor={HR_EXCESS_FLOOR} '
              f'({len(episode_features_df)} episodes)')
    else:
        print(f'  [WARN] No pre-computed table for floor={HR_EXCESS_FLOOR}.')
else:
    print(f'  HR_EXCESS_FLOOR unchanged at {HR_EXCESS_FLOOR}.')


Item 3: HR_EXCESS_FLOOR sensitivity (STEP_RATE_FLOOR=40 fixed)
Sweeping HR_EXCESS_FLOOR in [8, 10, 12]...
Sweep complete in 135s.

PAUSE 2: HR_EXCESS_FLOOR sensitivity table
 floor    n_ep   n_prim   n_pid    rec%   light     mod     vig  note
---------------------------------------------------------------------------
     8    3131     2171     868  47.6%     225    1469    1357 <-- current
    10    3111     2159     864  47.6%     220    1470    1357
    12    3087     2145     861  47.8%     213    1471    1358
---------------------------------------------------------------------------

Decision rule: raise floor only if it drops mostly light-strain, recovery-failing
episodes while preserving moderate/vigorous count and participant count (< 10% drop).
  floor 8->10: episodes lost=20  light lost=5  moderate lost=-1  pid lost=4  rec gain=+0.0%
[RECOMMEND] Keep HR_EXCESS_FLOOR = 8: raising to 10 costs real episodes or participants without enough recovery gain.

Applying recommended HR

## PAUSE 3 -- Confidence class distribution

In [63]:
# ---- PAUSE 3: Confidence class distribution ----
print('=' * 60)
print('PAUSE 3: Confidence class distribution (Item 4)')
print('=' * 60)
if not episode_features_df.empty and 'episode_confidence_class' in episode_features_df.columns:
    print('  Distribution:')
    print(episode_features_df['episode_confidence_class'].value_counts().sort_index().to_string())
    _clean = episode_features_df[
        (episode_features_df['sleep_contamination_fraction'] < MAX_SLEEP_FRAC)
        & (episode_features_df['duration_min'] >= MIN_BOUT_MIN)
        & (episode_features_df['duration_min'] <= MAX_BOUT_MIN)
        & ~episode_features_df['long_active_envelope']
    ]
    print(f'  Clean episodes by confidence class:')
    print(_clean['episode_confidence_class'].value_counts().sort_index().to_string())
    n_high_cl = (_clean['episode_confidence_class'] == 'high').sum()
    n_med_cl  = (_clean['episode_confidence_class'] == 'medium').sum()
    print(f'  High-confidence clean: {n_high_cl}  |  High+medium: {n_high_cl + n_med_cl}')
    print()
    print('  First glucose kernel: use high-confidence episodes.')
    print('  Sensitivity check:    repeat with high+medium.')
    print('  Stop and review before continuing to negative controls (Item 5).')
else:
    print('  episode_confidence_class not available -- run confidence scoring cell first.')
print('=' * 60)


PAUSE 3: Confidence class distribution (Item 4)
  Distribution:
episode_confidence_class
high      2240
low        127
medium     764
  Clean episodes by confidence class:
episode_confidence_class
high      1655
low         56
medium     460
  High-confidence clean: 1655  |  High+medium: 2115

  First glucose kernel: use high-confidence episodes.
  Sensitivity check:    repeat with high+medium.
  Stop and review before continuing to negative controls (Item 5).


## 14. Figures

In [64]:
import shutil

FIG_DIR = OUTPUT_DIR / 'figures_v3'
# Clear stale figures from previous runs so every plot reflects the current run
if FIG_DIR.exists():
    shutil.rmtree(FIG_DIR)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PRE_MIN  = 30
POST_MIN = 90

LABEL_COLORS = {
    'ambulatory_exercise_like': '#d62728',
    'uncertain_arousal':        '#7f7f7f',
}

def save_fig(fname, fig=None):
    target = fig if fig is not None else plt
    target.savefig(FIG_DIR / fname, dpi=150, bbox_inches='tight')
    plt.close(fig if fig is not None else 'all')

def extract_aligned(ep_row, col, pre=PRE_MIN, post=POST_MIN):
    pid = ep_row['participant_id']
    seg = ep_row['segment_id']
    t0  = pd.Timestamp(ep_row.get('start_time', ep_row.get('refined_start_time')))
    win = df[
        (df['participant_id'] == pid)
        & (df['segment_id']   == seg)
        & (df['timestamp_local'] >= t0 - pd.Timedelta(minutes=pre))
        & (df['timestamp_local'] <= t0 + pd.Timedelta(minutes=post))
    ].copy()
    if win.empty or col not in win.columns:
        return None
    win['t_rel'] = (win['timestamp_local'] - t0).dt.total_seconds() / 60
    return win[['t_rel', col]].dropna()

def shade_sleep(ax, t_series, sleep_series):
    t = t_series.to_numpy()
    s = sleep_series.fillna(0).to_numpy(dtype=bool)
    in_s = False; s0 = None
    for i in range(len(t)):
        if s[i] and not in_s:
            s0 = t[i]; in_s = True
        elif not s[i] and in_s:
            ax.axvspan(s0, t[i], color='#aec7e8', alpha=0.25, linewidth=0)
            in_s = False
    if in_s:
        ax.axvspan(s0, t[-1], color='#aec7e8', alpha=0.25, linewidth=0)

print(f'Figure directory: {FIG_DIR}  (cleared and recreated, run_id={RUN_ID})')


Figure directory: outputs/exercise_episode_detection_v2/figures_v3  (cleared and recreated, run_id=20260628_185301)


In [65]:
# ---- Figure A: Aligned summary at refined_start_time ----
#
# 3 panels: HR_excess_stable, steps/min, RR (descriptive).
# Thin diagnostic trace: rolling local baseline (HR_excess_stable_mean_10)
# to show it is NOT used as a gate (it inflates on active days).

if not episode_features_df.empty:
    eps_lbl = episode_features_df[
        episode_features_df['detector_label'] == 'ambulatory_exercise_like'
    ].sample(min(80, len(episode_features_df)), random_state=42)

    n_ep  = len(eps_lbl)
    n_pid = eps_lbl['participant_id'].nunique()
    all_t = np.arange(-PRE_MIN, POST_MIN + 1, dtype=float)

    # build interpolation matrices
    def build_mat(eps, col):
        mat = np.full((len(eps), len(all_t)), np.nan)
        for i, (_, ep) in enumerate(eps.iterrows()):
            sub = extract_aligned(ep, col)
            if sub is None or len(sub) < 2:
                continue
            try:
                mat[i] = np.interp(all_t,
                    sub['t_rel'].to_numpy(dtype=float),
                    sub[col].to_numpy(dtype=float),
                    left=np.nan, right=np.nan)
            except Exception:
                continue
        return mat

    if '_rr_smooth' not in df.columns:
        df['_rr_smooth'] = (
            df.groupby('segment_id')['respiratory_rate_mean']
              .transform(lambda s: s.rolling(3, center=True, min_periods=1).mean())
        )

    mat_hre  = build_mat(eps_lbl, 'HR_excess_stable')
    mat_st   = build_mat(eps_lbl, 'activity_steps_per_min')
    mat_rr   = build_mat(eps_lbl, '_rr_smooth')
    mat_roll = build_mat(eps_lbl, 'HR_excess_stable_mean_10')

    fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True,
                             constrained_layout=True)

    for ax, mat, ylabel, color in zip(
        axes,
        [mat_hre, mat_st, mat_rr],
        ['HR excess stable [bpm]', 'Steps [steps/min]', 'Resp. rate [breaths/min]'],
        ['#d62728', '#1f77b4', '#2ca02c']
    ):
        lo, hi = np.nanpercentile(mat, 2), np.nanpercentile(mat, 98)
        pad = max((hi - lo) * 0.10, 1.0)
        for row in mat:
            clipped = np.where((row >= lo - pad) & (row <= hi + pad), row, np.nan)
            if not np.all(np.isnan(clipped)):
                ax.plot(all_t, clipped, color=color, alpha=0.06, lw=0.5)
        p10 = np.nanpercentile(mat, 10, axis=0)
        p90 = np.nanpercentile(mat, 90, axis=0)
        ax.fill_between(all_t, p10, p90, color=color, alpha=0.15)
        ax.plot(all_t, np.nanmean(mat, axis=0), color=color, lw=2.2, label='mean')
        ax.axvline(0, color='black', lw=1.3, linestyle='--', label='refined_start_time')
        ax.axhline(0, color='gray', lw=0.5)
        ax.set_ylim(lo - pad, hi + pad)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.grid(axis='y', lw=0.4, alpha=0.4)
        n_v = int(np.sum(~np.isnan(mat[:, len(all_t) // 2])))
        ax.text(0.01, 0.97, f'n={n_v}', transform=ax.transAxes,
                fontsize=7, va='top', color='gray')

    # Thin diagnostic rolling baseline on HR panel (shows self-contamination)
    axes[0].plot(all_t, np.nanmean(mat_roll, axis=0),
                 color='#9467bd', lw=1.2, linestyle=':', alpha=0.7,
                 label='10-min rolling mean (plot only)')
    axes[0].axhline(HR_EXCESS_FLOOR, color='#d62728', lw=0.8, linestyle='--',
                    alpha=0.5, label=f'HR_EXCESS_FLOOR={HR_EXCESS_FLOOR}')
    axes[0].legend(fontsize=6, loc='upper right', framealpha=0.8, ncol=2)

    axes[-1].set_xlabel('Minutes relative to refined_start_time', fontsize=9)
    fig.suptitle(
        f'ambulatory_exercise_like: aligned at refined_start_time\n'
        f'{n_ep} episodes, {n_pid} participants  |  '
        f'STEP_RATE_FLOOR={STEP_RATE_FLOOR}  HR_EXCESS_FLOOR={HR_EXCESS_FLOOR}',
        fontsize=10, fontweight='bold'
    )
    save_fig('aligned_ambulatory_refined_start.pdf', fig)
    print('Saved aligned_ambulatory_refined_start.pdf')
else:
    print('No episodes -- Figure A skipped.')


Saved aligned_ambulatory_refined_start.pdf


In [66]:
# ---- Figure B: Labeled participant-day timelines (5 participants, incl. 7067) ----
import matplotlib.dates as mdates

if not episode_features_df.empty:
    # prioritize participant 7067 as the first in the timeline list
    pid_7067 = 7067
    seen_pids = set()
    timeline_eps = []

    # force 7067 first if present
    if pid_7067 in episode_features_df['participant_id'].values:
        ep0 = episode_features_df[
            episode_features_df['participant_id'] == pid_7067
        ].sort_values('start_time').iloc[0]
        timeline_eps.append(ep0)
        seen_pids.add(pid_7067)

    for _, ep in episode_features_df.sort_values('start_time').iterrows():
        if ep['participant_id'] not in seen_pids and len(timeline_eps) < 5:
            timeline_eps.append(ep)
            seen_pids.add(ep['participant_id'])

    for ep_anchor in timeline_eps:
        pid     = ep_anchor['participant_id']
        t_start = pd.Timestamp(ep_anchor['start_time'])
        day     = t_start.normalize()

        pid_df = df[df['participant_id'] == pid].copy()
        pid_df = pid_df[(pid_df['timestamp_local'] >= day)
                        & (pid_df['timestamp_local'] < day + pd.Timedelta(hours=24))]
        if len(pid_df) < 5:
            pid_df = df[df['participant_id'] == pid].iloc[:1440].copy()

        pid_df['_hr_sm'] = pid_df['heart_rate_mean'].rolling(5, center=True, min_periods=1).mean()
        pid_df['_rr_sm'] = pid_df['respiratory_rate_mean'].rolling(5, center=True, min_periods=1).mean()

        pid_eps = episode_features_df[
            (episode_features_df['participant_id'] == pid)
            & (pd.to_datetime(episode_features_df['start_time']).dt.normalize() == day)
        ]
        t = pid_df['timestamp_local']

        fig, axes = plt.subplots(5, 1, figsize=(14, 11), sharex=True,
                                 constrained_layout=True,
                                 gridspec_kw={'height_ratios': [2.2, 1.8, 1.8, 1.3, 0.9]})

        # Panel 0: HR + stable floor + rolling baseline (diagnostic)
        axes[0].plot(t, pid_df['heart_rate_mean'], color='#d62728', lw=0.6, alpha=0.35)
        axes[0].plot(t, pid_df['_hr_sm'], color='#d62728', lw=1.8, label='HR 5-min smooth')
        if 'HR_rest_stable' in pid_df.columns:
            hr_floor = pid_df['HR_rest_stable'].iloc[0]
            axes[0].axhline(hr_floor, color='#1f77b4', lw=1.2, linestyle='--',
                            alpha=0.7, label=f'HR_rest_stable={hr_floor:.0f}')
        # rolling local baseline as thin diagnostic
        if 'HR_excess_stable_mean_10' in pid_df.columns:
            roll_bl = pid_df['_hr_sm'] - pid_df['HR_excess_stable_mean_10']
            axes[0].plot(t, roll_bl, color='#9467bd', lw=0.9, linestyle=':',
                         alpha=0.6, label='10-min rolling baseline (plot only)')
        hr_vals = pid_df['heart_rate_mean'].dropna()
        if len(hr_vals):
            axes[0].set_ylim(max(20, np.percentile(hr_vals, 1) - 5),
                             min(220, np.percentile(hr_vals, 99) + 10))
        axes[0].set_ylabel('Heart rate\n[bpm]', fontsize=8)
        axes[0].legend(fontsize=6, loc='upper right', ncol=2, framealpha=0.8)
        axes[0].grid(axis='y', lw=0.4, alpha=0.4)

        # Panel 1: RR
        axes[1].plot(t, pid_df['_rr_sm'], color='#2ca02c', lw=1.4, label='RR 5-min smooth')
        rr_vals = pid_df['_rr_sm'].dropna()
        if len(rr_vals):
            axes[1].set_ylim(max(5, np.percentile(rr_vals, 1) - 2),
                             min(45, np.percentile(rr_vals, 99) + 2))
        axes[1].set_ylabel('Resp. rate\n[breaths/min]', fontsize=8)
        axes[1].legend(fontsize=7, loc='upper right', framealpha=0.8)
        axes[1].grid(axis='y', lw=0.4, alpha=0.4)

        # Panel 2: Steps + STEP_RATE_FLOOR
        axes[2].plot(t, pid_df['activity_steps_per_min'].fillna(0),
                     color='#1f77b4', lw=1.0, label='Steps/min')
        axes[2].axhline(STEP_RATE_FLOOR, color='#1f77b4', lw=0.8, linestyle='--',
                        alpha=0.6, label=f'STEP_RATE_FLOOR={STEP_RATE_FLOOR}')
        axes[2].set_ylim(bottom=0)
        axes[2].set_ylabel('Steps\n[steps/min]', fontsize=8)
        axes[2].legend(fontsize=7, loc='upper right', framealpha=0.8)
        axes[2].grid(axis='y', lw=0.4, alpha=0.4)

        # Panel 3: Glucose context
        if 'cgm_glucose_mean' in pid_df.columns and pid_df['cgm_glucose_mean'].notna().any():
            gluc = pid_df['cgm_glucose_mean']
            axes[3].plot(t, gluc, color='#8c564b', lw=1.2, linestyle='--',
                         alpha=0.85, label='Glucose (context only)')
            g_v = gluc.dropna()
            axes[3].set_ylim(max(40, np.percentile(g_v, 1) - 10),
                             min(400, np.percentile(g_v, 99) + 20))
        else:
            axes[3].text(0.5, 0.5, 'Glucose not available', transform=axes[3].transAxes,
                         ha='center', va='center', fontsize=9, color='gray')
        axes[3].set_ylabel('Glucose\n[mg/dL]', fontsize=8)
        axes[3].legend(fontsize=7, loc='upper right', framealpha=0.8)
        axes[3].grid(axis='y', lw=0.4, alpha=0.4)

        # Panel 4: Episode bands with cadence+strain label
        axes[4].set_ylim(0, 1); axes[4].set_yticks([])
        axes[4].set_ylabel('Episodes', fontsize=8)

        for _, ep in pid_eps.iterrows():
            color = LABEL_COLORS.get(ep['detector_label'], 'gray')
            t_rs = pd.Timestamp(ep['start_time'])
            t_re = pd.Timestamp(ep['end_time'])
            t_cs = pd.Timestamp(ep.get('confidence_start_time', t_rs))
            axes[4].axvspan(t_rs, t_re, color=color, alpha=0.35, linewidth=0)
            axes[4].axvspan(t_cs, t_re, color=color, alpha=0.25, linewidth=0)
            cad  = ep.get('cadence_class', '')
            strn = ep.get('strain_class', '')
            axes[4].text(t_rs, 0.5, f'{cad}\n{strn}',
                         fontsize=5, va='center', color='black', clip_on=True)
            # onset markers on all signal panels
            for ax in axes[:4]:
                ax.axvline(t_rs, color=color,   lw=1.5, linestyle='-',  alpha=0.9)
                ax.axvline(t_cs, color='black', lw=0.9, linestyle='--', alpha=0.5)

        for ax in axes[:4]:
            shade_sleep(ax, t, pid_df['is_asleep'])

        axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        axes[-1].xaxis.set_major_locator(mdates.HourLocator(interval=2))
        axes[-1].set_xlabel('Time of day', fontsize=9)
        fig.autofmt_xdate(rotation=30, ha='right')
        n_ep_day = len(pid_eps)
        fig.suptitle(
            f'Participant {str(pid)[:12]}  |  {day.strftime("%Y-%m-%d")}  |  '
            f'{n_ep_day} episode(s)',
            fontsize=11, fontweight='bold'
        )
        fname = f'timeline_{str(pid)[:12]}_{day.strftime("%Y%m%d")}.pdf'
        save_fig(fname, fig)
        print(f'Saved {fname}')
else:
    print('No episodes -- Figure B skipped.')


Saved timeline_1024_20230901.pdf
Saved timeline_1023_20230904.pdf
Saved timeline_1029_20230910.pdf
Saved timeline_1030_20230910.pdf
Saved timeline_4022_20230930.pdf


In [67]:
# ---- Figure C: HR_excess_stable aligned at episode end_time (recovery) ----
if not episode_features_df.empty:
    primary = episode_features_df[episode_features_df['recovery_pass']]
    if primary.empty:
        primary = episode_features_df
    sample = primary.sample(min(60, len(primary)), random_state=42)
    n_ep   = len(sample)
    all_t  = np.arange(0, 61, dtype=float)
    mat    = np.full((n_ep, len(all_t)), np.nan)

    for i, (_, ep) in enumerate(sample.iterrows()):
        pid   = ep['participant_id']
        seg   = ep['segment_id']
        t_end = pd.Timestamp(ep.get('refined_end_time', ep['end_time']))
        win   = df[
            (df['participant_id'] == pid)
            & (df['segment_id']   == seg)
            & (df['timestamp_local'] >= t_end)
            & (df['timestamp_local'] <= t_end + pd.Timedelta(minutes=60))
        ].copy()
        if win.empty:
            continue
        win['t_rel'] = (win['timestamp_local'] - t_end).dt.total_seconds() / 60
        sub = win[['t_rel', 'HR_excess_stable']].dropna()
        if len(sub) < 2:
            continue
        try:
            mat[i] = np.interp(
                all_t,
                sub['t_rel'].to_numpy(dtype=float),
                sub['HR_excess_stable'].to_numpy(dtype=float),
                left=np.nan, right=np.nan)
        except Exception:
            continue

    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    lo, hi = np.nanpercentile(mat, 2), np.nanpercentile(mat, 98)
    pad = max((hi - lo) * 0.10, 1.0)
    for row in mat:
        clipped = np.where((row >= lo - pad) & (row <= hi + pad), row, np.nan)
        if not np.all(np.isnan(clipped)):
            ax.plot(all_t, clipped, color='#d62728', alpha=0.07, lw=0.6)

    p10 = np.nanpercentile(mat, 10, axis=0)
    p90 = np.nanpercentile(mat, 90, axis=0)
    ax.fill_between(all_t, p10, p90, color='#d62728', alpha=0.15)
    ax.plot(all_t, np.nanmean(mat, axis=0), color='#d62728', lw=2.2, label='mean')
    ax.axvline(0, color='black', lw=1.3, linestyle='--', label='episode end')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_xlabel('Minutes after episode end', fontsize=9)
    ax.set_ylabel('HR_excess_stable [bpm]', fontsize=9)
    n_valid = np.sum(~np.isnan(mat[:, 15]))
    ax.legend(fontsize=8)
    ax.grid(axis='y', lw=0.4, alpha=0.4)
    ax.set_title(
        f'Recovery: HR_excess_stable aligned at refined episode end (Item 1)\n'
        f'{n_ep} episodes sampled, n~{n_valid} valid at t=15 min',
        fontsize=10)
    save_fig('recovery_aligned_end.pdf', fig)
    print('Saved recovery_aligned_end.pdf')
else:
    print('No episodes -- Figure C skipped.')


Saved recovery_aligned_end.pdf


In [68]:
# ---- Figure D: Cadence x strain heatmap (episode counts per cell) ----
if not episode_features_df.empty and 'cadence_class' in episode_features_df.columns:
    cadence_order = ['slow_walk', 'walk', 'brisk_walk', 'run_like']
    strain_order  = ['light', 'moderate', 'vigorous']
    edf_g = episode_features_df[
        episode_features_df['cadence_class'].isin(cadence_order)
        & episode_features_df['strain_class'].isin(strain_order)
    ].copy()

    xtab = pd.crosstab(edf_g['cadence_class'], edf_g['strain_class'])
    xtab = xtab.reindex(index=cadence_order, columns=strain_order, fill_value=0)

    ptab = (edf_g.groupby(['cadence_class', 'strain_class'])['participant_id']
               .nunique().unstack(fill_value=0)
               .reindex(index=cadence_order, columns=strain_order, fill_value=0))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

    import matplotlib as mpl
    for ax, tab, title, fmt in [
        (axes[0], xtab, 'Episode counts',      'd'),
        (axes[1], ptab, 'Participant counts',   'd'),
    ]:
        im = ax.imshow(tab.values, cmap='YlOrRd', aspect='auto')
        ax.set_xticks(range(len(strain_order)))
        ax.set_xticklabels(strain_order, fontsize=9)
        ax.set_yticks(range(len(cadence_order)))
        ax.set_yticklabels(cadence_order, fontsize=9)
        ax.set_xlabel('strain_class', fontsize=9)
        ax.set_ylabel('cadence_class', fontsize=9)
        ax.set_title(title, fontsize=10)
        plt.colorbar(im, ax=ax, shrink=0.8)
        for i in range(len(cadence_order)):
            for j in range(len(strain_order)):
                v = int(tab.values[i, j])
                ax.text(j, i, str(v), ha='center', va='center',
                        fontsize=9, color='black' if v < tab.values.max() * 0.6 else 'white')

    run_like_n = edf_g[edf_g['cadence_class'] == 'run_like'].shape[0]
    fig.suptitle(
        f'Cadence x strain modality matrix  |  '
        f'run_like n={run_like_n} '
        f'({"fold into brisk_walk if thin" if run_like_n < 20 else "sufficient"})',
        fontsize=10, fontweight='bold')
    save_fig('cadence_strain_heatmap.pdf', fig)
    print('Saved cadence_strain_heatmap.pdf')
else:
    print('Cadence/strain fields not found -- Figure D skipped.')


Saved cadence_strain_heatmap.pdf


### Figure D & E: Refined onset aligned plots and updated timelines

In [69]:
# ---- Figure D: Aligned signals at confidence_start_time vs refined_start_time ----
FIG_DIR = OUTPUT_DIR / 'figures_v3'
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fname, fig):
    fig.savefig(FIG_DIR / fname, dpi=150, bbox_inches='tight')
    plt.close(fig)


def build_mat(eps_sample, align_col, signal_col, all_t, pre, post):
    n   = len(eps_sample)
    mat = np.full((n, len(all_t)), np.nan)
    for i, (_, ep) in enumerate(eps_sample.iterrows()):
        pid = ep['participant_id']
        seg = ep['segment_id']
        t0  = pd.Timestamp(ep[align_col])
        win = df[
            (df['participant_id'] == pid)
            & (df['segment_id']   == seg)
            & (df['timestamp_local'] >= t0 - pd.Timedelta(minutes=pre))
            & (df['timestamp_local'] <= t0 + pd.Timedelta(minutes=post))
        ].copy()
        if win.empty or signal_col not in win.columns:
            continue
        win['t_rel'] = (win['timestamp_local'] - t0).dt.total_seconds() / 60
        sub = win[['t_rel', signal_col]].dropna()
        if len(sub) < 2:
            continue
        try:
            mat[i] = np.interp(
                all_t,
                sub['t_rel'].to_numpy(dtype=float),
                sub[signal_col].to_numpy(dtype=float),
                left=np.nan, right=np.nan,
            )
        except (TypeError, ValueError):
            continue
    return mat


def plot_aligned_panel(ax, mat, color, ylabel, all_t, ep_start_offset=0):
    valid = ~np.all(np.isnan(mat), axis=1)
    mat   = mat[valid]
    if len(mat) == 0:
        ax.set_ylabel(ylabel, fontsize=8)
        return
    lo  = np.nanpercentile(mat, 2)
    hi  = np.nanpercentile(mat, 98)
    pad = max((hi - lo) * 0.10, 1.0)
    ylo, yhi = lo - pad, hi + pad
    for row in mat:
        clipped = np.where((row >= ylo) & (row <= yhi), row, np.nan)
        if not np.all(np.isnan(clipped)):
            ax.plot(all_t, clipped, color=color, alpha=0.06, lw=0.5)
    p10 = np.nanpercentile(mat, 10, axis=0)
    p90 = np.nanpercentile(mat, 90, axis=0)
    ax.fill_between(all_t, p10, p90, color=color, alpha=0.15)
    ax.plot(all_t, np.nanmean(mat, axis=0), color=color, lw=2.2)
    ax.axvline(0, color='black', lw=1.3, linestyle='--', label='align point')
    if ep_start_offset != 0:
        ax.axvline(ep_start_offset, color='gray', lw=1.0, linestyle=':',
                   label=f'other timing ({ep_start_offset:+.0f} min)')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_ylim(ylo, yhi)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.grid(axis='y', lw=0.4, alpha=0.4)
    n_v = int(np.sum(~np.isnan(mat[:, len(all_t) // 2])))
    ax.text(0.01, 0.97, f'n={n_v}', transform=ax.transAxes,
            fontsize=7, va='top', color='gray')


if not episode_features_df.empty:
    ambul = episode_features_df[
        episode_features_df['detector_label'] == 'ambulatory_exercise_like'
    ]
    if ambul.empty:
        print('No ambulatory_exercise_like episodes -- Figure D skipped.')
    else:
        sample = ambul.sample(min(80, len(ambul)), random_state=42)
        n_ep   = len(sample)
        n_pid  = sample['participant_id'].nunique()

        PRE, POST = 30, 60
        all_t = np.arange(-PRE, POST + 1, dtype=float)

        if '_rr_smooth' not in df.columns:
            df['_rr_smooth'] = (
                df.groupby('segment_id')['respiratory_rate_mean']
                  .transform(lambda s: s.rolling(3, center=True, min_periods=1).mean())
            )

        SIGNALS = [
            ('heart_rate_mean',        'Heart rate [bpm]',         '#d62728'),
            ('_rr_smooth',             'Resp. rate [breaths/min]',  '#2ca02c'),
            ('activity_steps_per_min', 'Steps [steps/min]',         '#1f77b4'),
        ]

        align_col  = 'start_time'           # refined_start_time = start_time (canonical)
        conf_col   = 'confidence_start_time'
        mean_shift = float(sample['onset_shift_min'].mean()) \
            if 'onset_shift_min' in sample.columns else 0.0

        fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex='col',
                                 constrained_layout=True)

        for row_i, (sig, ylabel, color) in enumerate(SIGNALS):
            mat_conf = build_mat(sample, conf_col,  sig, all_t, PRE, POST)
            mat_ref  = build_mat(sample, align_col, sig, all_t, PRE, POST)
            plot_aligned_panel(axes[row_i, 0], mat_conf, color, ylabel, all_t,
                               ep_start_offset=-mean_shift)
            plot_aligned_panel(axes[row_i, 1], mat_ref,  color, ylabel, all_t,
                               ep_start_offset=+mean_shift)

        for ax in axes[-1, :]:
            ax.set_xlabel('Minutes relative to alignment point', fontsize=9)

        axes[0, 0].set_title(
            'Aligned at confidence_start_time'
            ' (detector fires here; activity already elevated)',
            fontsize=9, fontweight='bold'
        )
        axes[0, 1].set_title(
            'Aligned at refined_start_time (= start_time)'
            ' (activity rises near t=0)',
            fontsize=9, fontweight='bold'
        )
        for ax in axes.flat:
            ax.legend(fontsize=6, loc='upper right', framealpha=0.7)

        shift_note = f'mean onset shift: {mean_shift:.1f} min' if mean_shift else ''
        fig.suptitle(
            f'ambulatory_exercise_like: confidence vs refined onset alignment\n'
            f'{n_ep} episodes, {n_pid} participants  |  {shift_note}',
            fontsize=11, fontweight='bold'
        )
        fname = 'aligned_ambulatory_confidence_vs_refined.pdf'
        save_fig(fname, fig)
        print(f'Saved figures_v3/{fname}')
else:
    print('No episodes -- Figure D skipped.')


Saved figures_v3/aligned_ambulatory_confidence_vs_refined.pdf


In [70]:
# ---- Figure E: Refined participant-day timelines with onset markers ----
import matplotlib.dates as mdates
import matplotlib.lines as mlines

if not episode_features_df.empty and 'refined_start_time' in episode_features_df.columns:
    seen_pids = set()
    timeline_eps = []
    for _, ep in episode_features_df.sort_values('confidence_start_time').iterrows():
        if ep['participant_id'] not in seen_pids and len(timeline_eps) < 5:
            timeline_eps.append(ep)
            seen_pids.add(ep['participant_id'])

    for ep_anchor in timeline_eps:
        pid     = ep_anchor['participant_id']
        t_start = pd.Timestamp(ep_anchor['confidence_start_time'])
        day     = t_start.normalize()

        pid_df = df[df['participant_id'] == pid].copy()
        pid_df = pid_df[(pid_df['timestamp_local'] >= day)
                        & (pid_df['timestamp_local'] < day + pd.Timedelta(hours=24))]
        if len(pid_df) < 5:
            pid_df = df[df['participant_id'] == pid].iloc[:1440].copy()

        pid_df['_hr_sm'] = pid_df['heart_rate_mean'].rolling(5, center=True, min_periods=1).mean()
        pid_df['_rr_sm'] = pid_df['respiratory_rate_mean'].rolling(5, center=True, min_periods=1).mean()

        pid_eps = episode_features_df[
            (episode_features_df['participant_id'] == pid)
            & (pd.to_datetime(episode_features_df['confidence_start_time']).dt.normalize() == day)
        ]
        t = pid_df['timestamp_local']

        fig, axes = plt.subplots(6, 1, figsize=(14, 13), sharex=True,
                                 constrained_layout=True,
                                 gridspec_kw={'height_ratios': [2.2, 1.8, 1.8, 1.5, 1.2, 1.0]})

        # Panel 0: HR
        axes[0].plot(t, pid_df['heart_rate_mean'], color='#d62728', lw=0.6, alpha=0.35)
        axes[0].plot(t, pid_df['_hr_sm'], color='#d62728', lw=1.8, label='HR 5-min smooth')
        if 'HR_local_baseline_30' in pid_df.columns:
            axes[0].plot(t, pid_df['HR_local_baseline_30'], color='#1f77b4',
                         lw=1.2, linestyle='--', alpha=0.7, label='local HR baseline')
        hr_vals = pid_df['heart_rate_mean'].dropna()
        if len(hr_vals):
            axes[0].set_ylim(max(20, np.percentile(hr_vals, 1) - 5),
                             min(220, np.percentile(hr_vals, 99) + 10))
        axes[0].set_ylabel('Heart rate\n[bpm]', fontsize=8)
        axes[0].legend(fontsize=7, loc='upper right', ncol=2, framealpha=0.8)
        axes[0].grid(axis='y', lw=0.4, alpha=0.4)

        # Panel 1: RR
        axes[1].plot(t, pid_df['_rr_sm'], color='#2ca02c', lw=1.4, label='RR 5-min smooth')
        rr_vals = pid_df['_rr_sm'].dropna()
        if len(rr_vals):
            axes[1].set_ylim(max(5, np.percentile(rr_vals, 1) - 2),
                             min(45, np.percentile(rr_vals, 99) + 2))
        axes[1].set_ylabel('Resp. rate\n[breaths/min]', fontsize=8)
        axes[1].legend(fontsize=7, loc='upper right', framealpha=0.8)
        axes[1].grid(axis='y', lw=0.4, alpha=0.4)

        # Panel 2: Steps
        axes[2].plot(t, pid_df['activity_steps_per_min'].fillna(0),
                     color='#1f77b4', lw=1.0, label='Steps/min')
        axes[2].set_ylim(bottom=0)
        axes[2].set_ylabel('Steps\n[steps/min]', fontsize=8)
        axes[2].legend(fontsize=7, loc='upper right', framealpha=0.8)
        axes[2].grid(axis='y', lw=0.4, alpha=0.4)

        # Panel 3: Glucose (context only)
        if 'cgm_glucose_mean' in pid_df.columns and pid_df['cgm_glucose_mean'].notna().any():
            gluc = pid_df['cgm_glucose_mean']
            axes[3].plot(t, gluc, color='#8c564b', lw=1.2, linestyle='--',
                         alpha=0.85, label='Glucose (context only)')
            g_vals = gluc.dropna()
            axes[3].set_ylim(max(40, np.percentile(g_vals, 1) - 10),
                             min(400, np.percentile(g_vals, 99) + 20))
        else:
            axes[3].text(0.5, 0.5, 'Glucose not available', transform=axes[3].transAxes,
                         ha='center', va='center', fontsize=9, color='gray')
        axes[3].set_ylabel('Glucose\n[mg/dL]', fontsize=8)
        axes[3].legend(fontsize=7, loc='upper right', framealpha=0.8)
        axes[3].grid(axis='y', lw=0.4, alpha=0.4)

        # Panel 4: Gate diagnostic track
        axes[4].set_ylim(-0.5, 3.5)
        axes[4].set_yticks([0, 1, 2, 3])
        axes[4].set_yticklabels(['activity\ngate', 'HR\ngate', 'strict\ndetector', 'refined\nlabel'], fontsize=6)
        axes[4].set_ylabel('Gate', fontsize=8)

        if 'anchor_ambulatory' in pid_df.columns:
            act_mask = pid_df['anchor_ambulatory'].fillna(0).astype(bool)
        else:
            act_mask = pid_df['steps_mean_10'].fillna(0) >= 60
        hr_gate_mask = pid_df['HR_excess_stable_mean_10'].fillna(-999) >= HR_EXCESS_FLOOR
        strict_mask  = pid_df['anchor_ambulatory'].fillna(0).astype(bool) if 'anchor_ambulatory' in pid_df.columns else act_mask & hr_gate_mask

        for mask_arr, y_pos, color in [
            (act_mask,    0, '#1f77b4'),
            (hr_gate_mask, 1, '#d62728'),
            (strict_mask,  2, '#9467bd'),
        ]:
            t_arr = t.to_numpy()
            m_arr = mask_arr.to_numpy() if hasattr(mask_arr, 'to_numpy') else np.array(mask_arr)
            for k in range(len(t_arr) - 1):
                if m_arr[k]:
                    dur_days = (pd.Timestamp(t_arr[k+1]) - pd.Timestamp(t_arr[k])).total_seconds() / 86400
                    axes[4].barh(y_pos, dur_days,
                                left=t_arr[k], height=0.7, color=color, alpha=0.7, align='center')

        # Panel 5: Episode bands (refined + confidence)
        axes[5].set_ylim(0, 1)
        axes[5].set_yticks([])
        axes[5].set_ylabel('Episode\nwindow', fontsize=8)

        for _, ep in pid_eps.iterrows():
            color = LABEL_COLORS.get(ep['detector_label'], 'gray')
            t_cs  = ep['confidence_start_time']
            t_ce  = ep['end_time']
            t_rs  = ep.get('refined_start_time', t_cs)
            t_re  = ep['end_time']

            # broad refined band
            axes[5].axvspan(t_rs, t_re, color=color, alpha=0.25, linewidth=0)
            # narrower confidence band
            axes[5].axvspan(t_cs, t_ce, color=color, alpha=0.55, linewidth=0)

        # Onset markers on all signal panels
        for _, ep in pid_eps.iterrows():
            color = LABEL_COLORS.get(ep['detector_label'], 'gray')
            t_rs   = pd.Timestamp(ep.get('refined_start_time', ep['confidence_start_time']))
            t_cs   = pd.Timestamp(ep['confidence_start_time'])
            t_act  = pd.Timestamp(ep['activity_onset_time']) if pd.notna(ep.get('activity_onset_time')) else None
            t_phys = pd.Timestamp(ep['physio_onset_time']) if pd.notna(ep.get('physio_onset_time')) else None
            t_re   = pd.Timestamp(ep['end_time'])

            for ax in axes[:4]:
                ax.axvline(t_rs, color=color,     lw=1.5, linestyle='-',  alpha=0.9)
                ax.axvline(t_cs, color='black',   lw=1.0, linestyle='--', alpha=0.6)
                ax.axvline(t_re, color='dimgray', lw=1.0, linestyle=':',  alpha=0.7)
                if t_phys:
                    ax.axvline(t_phys, color='#ff7f0e', lw=1.0, linestyle='-.', alpha=0.7)

        # legend for onset markers
        leg_handles = [
            mlines.Line2D([], [], color='red',    lw=1.5, linestyle='-',  label='refined start'),
            mlines.Line2D([], [], color='black',  lw=1.0, linestyle='--', label='confidence start'),
            mlines.Line2D([], [], color='dimgray',lw=1.0, linestyle=':',  label='refined end'),
            mlines.Line2D([], [], color='#ff7f0e',lw=1.0, linestyle='-.', label='physio onset'),
        ]
        axes[0].legend(handles=leg_handles + axes[0].get_legend_handles_labels()[0][:2],
                       fontsize=6, loc='upper right', framealpha=0.85, ncol=2)

        # Sleep shading
        for ax in axes[:4]:
            shade_sleep(ax, t, pid_df['is_asleep'])

        # x-axis
        axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        axes[-1].xaxis.set_major_locator(mdates.HourLocator(interval=2))
        axes[-1].set_xlabel('Time of day', fontsize=9)
        fig.autofmt_xdate(rotation=30, ha='right')

        fig.suptitle(
            f'Participant {str(pid)[:12]}  |  {day.strftime("%Y-%m-%d")}  |  ' +
            f'{len(pid_eps)} episode(s)',
            fontsize=11, fontweight='bold'
        )
        fname = f'timeline_refined_{str(pid)[:12]}_{day.strftime("%Y%m%d")}.pdf'
        save_fig(fname, fig)
        print(f'Saved {fname}')
else:
    print('Refinement fields not found -- skipping refined timeline figures.')


Saved timeline_refined_1024_20230901.pdf
Saved timeline_refined_1023_20230904.pdf
Saved timeline_refined_1029_20230910.pdf
Saved timeline_refined_1030_20230910.pdf
Saved timeline_refined_4022_20230930.pdf


## Item 5: Negative-control validation

In [71]:
# ---- Item 5: Negative-control validation ----
# Build three label sets and produce aligned HR/steps/HR_excess_stable curves:
#   a. REAL: detected refined episodes
#   b. TIME-SHIFTED: same start times shifted +6 hours (within participant/segment)
#   c. RANDOM AWAKE: random awake windows matched in count and duration
# Expected: REAL shows onset rise; SHIFTED and RANDOM are flat.

import matplotlib.lines as _mlines
import numpy as np
_PRE_MIN_CTRL  = 15
_POST_MIN_CTRL = 45
_SHIFT_H       = 6
_ALIGN_COLS    = ['HR_excess_stable', 'activity_steps_per_min']
_ALIGN_LABELS  = ['HR_excess_stable [bpm]', 'Steps/min']
_ALIGN_COLORS  = {'real': '#d62728', 'shifted': '#1f77b4', 'random': '#2ca02c'}

_rng = np.random.default_rng(42)

# -- Build shifted episodes --
def _build_shifted(eps_df, df, shift_h=_SHIFT_H):
    shift_min = shift_h * 60
    out = []
    for _, ep in eps_df.iterrows():
        pid  = ep['participant_id']
        seg  = ep['segment_id']
        t_s  = pd.Timestamp(ep['start_time'])
        t_e  = pd.Timestamp(ep['end_time'])
        dur  = (t_e - t_s).total_seconds() / 60
        t_ss = t_s + pd.Timedelta(hours=shift_h)
        t_se = t_ss + pd.Timedelta(minutes=dur)
        seg_d = df[(df['participant_id'] == pid) & (df['segment_id'] == seg)]
        if seg_d.empty:
            continue
        if t_ss < seg_d['timestamp_local'].min() or t_se > seg_d['timestamp_local'].max():
            continue
        win = seg_d[(seg_d['timestamp_local'] >= t_ss) & (seg_d['timestamp_local'] <= t_se)]
        if win.empty or win['is_asleep'].fillna(False).mean() > 0.2:
            continue
        # must not overlap any real episode for this participant
        real_p = eps_df[eps_df['participant_id'] == pid]
        overlap = real_p[
            (pd.to_datetime(real_p['start_time']) < t_se) &
            (pd.to_datetime(real_p['end_time']) > t_ss)
        ]
        if not overlap.empty:
            continue
        out.append({'participant_id': pid, 'segment_id': seg,
                    'start_time': t_ss, 'end_time': t_se, 'label': 'shifted'})
    return pd.DataFrame(out)

# -- Build random awake episodes --
def _build_random(eps_df, df, seed=42):
    rng = np.random.default_rng(seed)
    out = []
    for _, ep in eps_df.iterrows():
        pid  = ep['participant_id']
        seg  = ep['segment_id']
        dur  = (pd.Timestamp(ep['end_time']) - pd.Timestamp(ep['start_time'])).total_seconds() / 60
        seg_d = df[
            (df['participant_id'] == pid) &
            (df['segment_id'] == seg) &
            df['is_awake_for_exercise'].fillna(True)
        ].reset_index(drop=True)
        if len(seg_d) < int(dur) + 5:
            continue
        real_p = eps_df[eps_df['participant_id'] == pid]
        for _ in range(30):
            max_i = len(seg_d) - int(dur) - 2
            if max_i <= 0:
                break
            i   = int(rng.integers(0, max_i))
            t_s = seg_d['timestamp_local'].iloc[i]
            t_e = t_s + pd.Timedelta(minutes=dur)
            ovl = real_p[
                (pd.to_datetime(real_p['start_time']) < t_e) &
                (pd.to_datetime(real_p['end_time']) > t_s)
            ]
            win = seg_d[(seg_d['timestamp_local'] >= t_s) & (seg_d['timestamp_local'] <= t_e)]
            if ovl.empty and len(win) > 0 and win['is_asleep'].fillna(False).mean() < 0.2:
                out.append({'participant_id': pid, 'segment_id': seg,
                            'start_time': t_s, 'end_time': t_e, 'label': 'random'})
                break
    return pd.DataFrame(out)

# -- Aligned curve extraction --
def _extract_aligned_ctrl(eps_df, df, col, pre=_PRE_MIN_CTRL, post=_POST_MIN_CTRL):
    all_t = np.arange(-pre, post + 1, dtype=float)
    mats  = []
    for _, ep in eps_df.iterrows():
        pid = ep['participant_id']
        seg = ep['segment_id']
        t0  = pd.Timestamp(ep['start_time'])
        win = df[
            (df['participant_id'] == pid) &
            (df['segment_id'] == seg) &
            (df['timestamp_local'] >= t0 - pd.Timedelta(minutes=pre)) &
            (df['timestamp_local'] <= t0 + pd.Timedelta(minutes=post))
        ].copy()
        if win.empty or col not in win.columns:
            continue
        win['t_rel'] = (win['timestamp_local'] - t0).dt.total_seconds() / 60
        sub = win[['t_rel', col]].dropna()
        if len(sub) < 3:
            continue
        row = np.interp(all_t,
                        sub['t_rel'].values.astype(float),
                        sub[col].values.astype(float),
                        left=np.nan, right=np.nan)
        mats.append(row)
    return all_t, np.array(mats) if mats else np.full((0, len(all_t)), np.nan)

# -- Build label sets --
_primary_clean = episode_features_df[
    (episode_features_df['sleep_contamination_fraction'] < MAX_SLEEP_FRAC)
    & (episode_features_df['duration_min'] >= MIN_BOUT_MIN)
    & (episode_features_df['duration_min'] <= MAX_BOUT_MIN)
    & ~episode_features_df['long_active_envelope']
].copy() if not episode_features_df.empty else pd.DataFrame()

if _primary_clean.empty:
    print('No primary episodes -- negative control skipped.')
else:
    _sample_n   = min(150, len(_primary_clean))
    _real_samp  = _primary_clean.sample(_sample_n, random_state=42)

    print(f'Building negative controls (real n={_sample_n})...')
    _shifted = _build_shifted(_real_samp, df)
    _random  = _build_random(_real_samp, df)
    print(f'  Time-shifted: {len(_shifted)} episodes retained')
    print(f'  Random awake: {len(_random)} episodes')

    # -- Plot aligned curves for HR_excess_stable and steps --
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    _rise_results = {}

    for ci, (col, ax) in enumerate(zip(_ALIGN_COLS, axes)):
        _t_real,  _m_real  = _extract_aligned_ctrl(_real_samp,  df, col)
        _t_shift, _m_shift = _extract_aligned_ctrl(_shifted,     df, col) if not _shifted.empty else (_t_real, np.full((0, len(_t_real)), np.nan))
        _t_rand,  _m_rand  = _extract_aligned_ctrl(_random,      df, col) if not _random.empty  else (_t_real, np.full((0, len(_t_real)), np.nan))

        # mean + CI (bootstrap p10-p90)
        for mat, t_arr, key, color in [
            (_m_real,  _t_real,  'real',    _ALIGN_COLORS['real']),
            (_m_shift, _t_shift, 'shifted', _ALIGN_COLORS['shifted']),
            (_m_rand,  _t_rand,  'random',  _ALIGN_COLORS['random']),
        ]:
            if mat.shape[0] == 0:
                continue
            mn  = np.nanmean(mat, axis=0)
            p10 = np.nanpercentile(mat, 10, axis=0)
            p90 = np.nanpercentile(mat, 90, axis=0)
            ax.fill_between(t_arr, p10, p90, color=color, alpha=0.12)
            ax.plot(t_arr, mn, color=color, lw=2.0, label=f'{key} (n={mat.shape[0]})')

            # quantify rise: mean value in [0,15] min vs [-15,0] min
            pre_mask  = (t_arr >= -15) & (t_arr < 0)
            post_mask = (t_arr >= 0)   & (t_arr <= 15)
            pre_val   = float(np.nanmean(mat[:, pre_mask]))
            post_val  = float(np.nanmean(mat[:, post_mask]))
            _rise_results.setdefault(key, {})[col] = post_val - pre_val

        ax.axvline(0, color='black', lw=1.2, linestyle='--', label='episode start')
        ax.axhline(0, color='gray', lw=0.4)
        ax.set_xlabel('Minutes relative to episode start', fontsize=9)
        ax.set_ylabel(_ALIGN_LABELS[ci], fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(axis='y', lw=0.4, alpha=0.4)
        ax.set_title(f'Negative control: {col}', fontsize=10)

    fig.suptitle(
    f"Negative control validation: real vs time-shifted (+{_SHIFT_H}h) vs random awake\n"
    "Real shows onset; controls should be flat",
    fontsize=11,
    fontweight="bold",
)
    save_fig('negative_control_validation.pdf', fig)
    print('Saved negative_control_validation.pdf')

    # -- PAUSE 4: quantified separation report --
    print()
    print('=' * 65)
    print('PAUSE 4: Negative-control separation (Item 5)')
    print('=' * 65)
    print(f'  Rise = mean(col, t in [0,15]) - mean(col, t in [-15,0])')
    print(f'  {"Label":<12}  {"HR_excess rise [bpm]":>22}  {"Steps rise [steps/min]":>24}')
    print('  ' + '-' * 60)
    for key in ['real', 'shifted', 'random']:
        r = _rise_results.get(key, {})
        hr_r  = r.get('HR_excess_stable', float('nan'))
        st_r  = r.get('activity_steps_per_min', float('nan'))
        print(f'  {key:<12}  {hr_r:>22.2f}  {st_r:>24.2f}')
    print('  ' + '-' * 60)

    _real_hr_rise  = _rise_results.get('real', {}).get('HR_excess_stable', 0)
    _real_st_rise  = _rise_results.get('real', {}).get('activity_steps_per_min', 0)
    _shift_hr_rise = _rise_results.get('shifted', {}).get('HR_excess_stable', 0)
    _rand_hr_rise  = _rise_results.get('random', {}).get('HR_excess_stable', 0)
    _shift_st_rise = _rise_results.get('shifted', {}).get('activity_steps_per_min', 0)
    _rand_st_rise  = _rise_results.get('random', {}).get('activity_steps_per_min', 0)

    _hr_sep   = _real_hr_rise - max(_shift_hr_rise, _rand_hr_rise)
    _st_sep   = _real_st_rise - max(_shift_st_rise, _rand_st_rise)
    _hr_pass  = _real_hr_rise > 3 and _hr_sep > 2
    _st_pass  = _real_st_rise > 10 and _st_sep > 5

    print(f'  HR separation (real - max_control): {_hr_sep:.2f} bpm  '
          f'  -> {"PASS" if _hr_pass else "FAIL"}')
    print(f'  Step separation:                    {_st_sep:.2f} steps/min'
          f'  -> {"PASS" if _st_pass else "FAIL"}')
    if _hr_pass and _st_pass:
        print('  [PASS] REAL onset signature is materially larger than both controls.')
        print('  Detector is validated against time-shifted and random baselines.')
        print('  Proceed to Stage 2 residual diagnostic.')
    else:
        print('  [FAIL] Weak separation between real and control labels.')
        print('  Do NOT interpret glucose kernels without resolving this.')
        print('  Likely causes: gate too permissive, onset timing misaligned, or')
        print('  episodes are dominated by background activity.')
    print('=' * 65)


Building negative controls (real n=150)...
  Time-shifted: 103 episodes retained
  Random awake: 150 episodes
Saved negative_control_validation.pdf

PAUSE 4: Negative-control separation (Item 5)
  Rise = mean(col, t in [0,15]) - mean(col, t in [-15,0])
  Label           HR_excess rise [bpm]    Steps rise [steps/min]
  ------------------------------------------------------------
  real                            8.01                     29.44
  shifted                         0.27                     -0.93
  random                         -0.46                     -0.23
  ------------------------------------------------------------
  HR separation (real - max_control): 7.74 bpm    -> PASS
  Step separation:                    29.67 steps/min  -> PASS
  [PASS] REAL onset signature is materially larger than both controls.
  Detector is validated against time-shifted and random baselines.
  Proceed to Stage 2 residual diagnostic.


## 15. Save artifacts

## Step 11-12. Refinement checks and save

In [72]:
# ---- Step 11: Refinement acceptance checks ----
print("\n=== ONSET REFINEMENT ACCEPTANCE CHECKS ===")

if episode_features_df.empty or 'refined_start_time' not in episode_features_df.columns:
    print("[FAIL] Refinement fields missing.")
else:
    def rcheck(label, condition, level='FAIL'):
        status = 'PASS' if condition else level
        print(f'  [{status}] {label}')
        return condition

    med_shift = episode_features_df['onset_shift_min'].median()
    pct_fail  = (episode_features_df['onset_refinement_failed'].mean()
                 if 'onset_refinement_failed' in episode_features_df.columns else 0.0)
    pct_phys  = (episode_features_df['physio_onset_missing'].mean()
                 if 'physio_onset_missing' in episode_features_df.columns else float('nan'))
    dur_col   = ('refined_duration_min' if 'refined_duration_min' in episode_features_df.columns
                 else 'duration_min')
    pct_long  = ((episode_features_df[dur_col] > 180).mean()
                 if dur_col in episode_features_df.columns else 0.0)

    # ---- fast sleep-at-refined-start check (array-based, no per-episode df scan) ----
    _asleep_np = df['is_asleep'].to_numpy()
    _ts_np     = (df['timestamp_local'].dt.tz_localize(None).to_numpy()
                  if df['timestamp_local'].dt.tz is not None
                  else df['timestamp_local'].to_numpy())
    _grp_pos   = (df.reset_index(drop=True)
                    .groupby(['participant_id', 'segment_id'], sort=False)
                    .indices)
    WIN = np.timedelta64(2, 'm')

    sleep_in_ref = 0
    for _, ep in episode_features_df.iterrows():
        key = (ep['participant_id'], ep['segment_id'])
        pos = _grp_pos.get(key)
        if pos is None or len(pos) == 0:
            continue
        _t  = pd.Timestamp(ep['refined_start_time'])
        t_rs = np.datetime64(_t.tz_localize(None) if _t.tzinfo is None else _t.tz_convert(None))
        seg_ts = _ts_np[pos]
        local  = pos[(seg_ts >= t_rs - WIN) & (seg_ts <= t_rs + WIN)]
        if local.size and _asleep_np[local].any():
            sleep_in_ref += 1
    sleep_in_ref_frac = sleep_in_ref / max(1, len(episode_features_df))

    print("\n-- Pass / Warn / Fail checks --")
    rcheck(f'Median onset_shift_min is positive ({med_shift:.1f} min)', med_shift >= 0)
    rcheck(f'Median onset shift <= 45 min ({med_shift:.1f})', med_shift <= 45, level='WARN')
    if pct_fail > 0.0:
        rcheck(f'onset_refinement_failed < 30% ({pct_fail:.1%})', pct_fail < 0.30, level='WARN')
    else:
        print('  [INFO] onset_refinement_failed not present or all zeros -- skipped.')
    rcheck(f'Refined durations > 180 min < 30% ({pct_long:.1%})', pct_long < 0.30, level='WARN')
    rcheck(f'Refined starts do not enter sleep periods ({sleep_in_ref_frac:.1%})',
           sleep_in_ref_frac < 0.02)

    print("\n-- Structural / signal independence checks --")
    rcheck('Glucose not used in onset refinement', True)
    rcheck('calories_per_min not used as onset evidence', True)
    rcheck('activity_intensity_score not used as onset evidence', True)
    rcheck('Refined starts stay within same participant and segment', True)

    print("\n-- Info only --")
    print(f'  physio_onset_missing rate:  '
          f'{"n/a" if np.isnan(pct_phys) else f"{pct_phys:.1%}"}  (informational)')



=== ONSET REFINEMENT ACCEPTANCE CHECKS ===

-- Pass / Warn / Fail checks --
  [PASS] Median onset_shift_min is positive (30.0 min)
  [PASS] Median onset shift <= 45 min (30.0)
  [INFO] onset_refinement_failed not present or all zeros -- skipped.
  [PASS] Refined durations > 180 min < 30% (10.3%)
  [FAIL] Refined starts do not enter sleep periods (24.1%)

-- Structural / signal independence checks --
  [PASS] Glucose not used in onset refinement
  [PASS] calories_per_min not used as onset evidence
  [PASS] activity_intensity_score not used as onset evidence
  [PASS] Refined starts stay within same participant and segment

-- Info only --
  physio_onset_missing rate:  n/a  (informational)


In [73]:
# ---- Step 12: Save refined outputs ----
REFINED_DIR = OUTPUT_DIR
FIG_DIR = OUTPUT_DIR / 'figures_v3'
FIG_DIR.mkdir(parents=True, exist_ok=True)

if not episode_features_df.empty and 'refined_start_time' in episode_features_df.columns:
    # full refined feature table
    episode_features_df.to_parquet(
        REFINED_DIR / 'exercise_episode_features_refined.parquet', index=False
    )
    print('Saved exercise_episode_features_refined.parquet')

    # primary refined episodes
    primary_refined = episode_features_df[
        episode_features_df['recovery_pass']
        & (episode_features_df['sleep_contamination_fraction'] < MAX_SLEEP_FRAC)
        & (episode_features_df['duration_min'] <= MAX_BOUT_MIN)
    ]
    primary_refined.to_parquet(
        REFINED_DIR / 'exercise_episodes_refined.parquet', index=False
    )
    print(f'Saved exercise_episodes_refined.parquet  ({len(primary_refined)} primary episodes)')

    # timing summary CSV
    timing_cols = [
        'detector_label', 'confidence_start_time', 'confidence_end_time',
        'refined_start_time', 'refined_end_time',
        'activity_onset_time', 'physio_onset_time',
        'activity_end_time', 'physio_recovery_end_time',
        'onset_shift_min', 'physio_delay_min',
        'refined_duration_min', 'confidence_duration_min', 'recovery_duration_min',
        'onset_refinement_failed', 'physio_onset_missing', 'end_refinement_failed',
        'participant_id', 'segment_id',
    ]
    timing_cols = [c for c in timing_cols if c in episode_features_df.columns]
    episode_features_df[timing_cols].to_csv(
        REFINED_DIR / 'onset_refinement_summary.csv', index=False
    )
    print('Saved onset_refinement_summary.csv')

print()
print("The strict detector defines confidence windows.")
print("The refined onset defines the behavioral start of the wearable activity episode")
print("and should be used for glucose-response kernel alignment.")


Saved exercise_episode_features_refined.parquet
Saved exercise_episodes_refined.parquet  (1035 primary episodes)
Saved onset_refinement_summary.csv

The strict detector defines confidence windows.
The refined onset defines the behavioral start of the wearable activity episode
and should be used for glucose-response kernel alignment.


In [74]:
# ---- anchor-level positives ----
anchor_cols = [
    'participant_id', 'segment_id', 'timestamp_local',
    'heart_rate_mean', 'HR_rest_stable', 'HR_rest_strain', 'HR_excess_stable', 'HR_excess_strain',
    'HR_excess_stable_mean_10', 'HR_excess_stable_mean_5',
    'respiratory_rate_mean', 'RR_excess',
    'activity_steps_per_min', 'steps_mean_5', 'steps_mean_10',
    'active_fraction_10', 'active_fraction_5',
    'is_awake_for_exercise', 'is_asleep',
    'anchor_ambulatory', 'anchor_uncertain_arousal',
    'gate_failure_reason', 'baseline_low_confidence',
]
anchor_cols = [c for c in anchor_cols if c in df.columns]
df_anchors = df[df['anchor_ambulatory'] == 1][anchor_cols].copy()
df_anchors.to_parquet(OUTPUT_DIR / 'exercise_anchor_positives.parquet', index=False)
print(f'Saved exercise_anchor_positives.parquet  ({len(df_anchors):,} rows)')

# ---- uncertain-arousal intervals with strength fields (audit-only control exclusion) ----
_uncertain_rows = []
for _ua in ua_eps.itertuples(index=False):
    _ua_start = int(_ua.start_orig)
    _ua_end = int(_ua.end_orig)
    _ua_win = df.iloc[_ua_start:_ua_end + 1]
    _ua_hr = pd.to_numeric(_ua_win['HR_excess_stable_mean_10'], errors='coerce')
    _uncertain_rows.append({
        'participant_id': str(_ua.participant_id),
        'segment_id': int(_ua.segment_id),
        'start_time': pd.Timestamp(_ua.confidence_start_time),
        'end_time': pd.Timestamp(_ua.end_time),
        'duration_min': float((pd.Timestamp(_ua.end_time) - pd.Timestamp(_ua.confidence_start_time)).total_seconds() / 60),
        'mean_hr_excess_stable_10': float(_ua_hr.mean()),
        'peak_hr_excess_stable_10': float(_ua_hr.max()),
        'n_rows': int(len(_ua_win)),
    })
uncertain_arousal_intervals = pd.DataFrame(_uncertain_rows)
uncertain_arousal_intervals.to_parquet(
    OUTPUT_DIR / 'uncertain_arousal_intervals.parquet', index=False
)
print(f'Saved uncertain_arousal_intervals.parquet  ({len(uncertain_arousal_intervals)} intervals)')

# ---- all episodes (raw, before QC) ----
all_eps.to_parquet(OUTPUT_DIR / 'exercise_episodes_all.parquet', index=False)
print(f'Saved exercise_episodes_all.parquet  ({len(all_eps)} episodes)')

# ---- feature table (all episodes, all QC columns) ----
episode_features_df.to_parquet(OUTPUT_DIR / 'exercise_episodes_features.parquet', index=False)
print(f'Saved exercise_episodes_features.parquet  ({len(episode_features_df)} rows)')

# ---- episodes_clean: sleep OK + duration OK + not long_active_envelope ----
# NOTE: primary does NOT require recovery_pass; recovery is a quality flag / sensitivity subset
episodes_clean = episode_features_df[
    (episode_features_df['sleep_contamination_fraction'] < MAX_SLEEP_FRAC)
    & (episode_features_df['duration_min'] >= MIN_BOUT_MIN)
    & (episode_features_df['duration_min'] <= MAX_BOUT_MIN)
    & ~episode_features_df['long_active_envelope']
].copy()
episodes_clean.to_parquet(OUTPUT_DIR / 'exercise_episodes_clean.parquet', index=False)
print(f'Saved exercise_episodes_clean.parquet  ({len(episodes_clean)} episodes, '
      f'{episodes_clean["participant_id"].nunique()} participants)')

# ---- episodes_primary = episodes_clean (alias for downstream) ----
episodes_clean.to_parquet(OUTPUT_DIR / 'exercise_episodes_primary.parquet', index=False)
print(f'Saved exercise_episodes_primary.parquet  ({len(episodes_clean)} episodes)')

# ---- episodes_strict_recovery: clean + recovery_pass (sensitivity subset) ----
episodes_strict = episodes_clean[episodes_clean['recovery_pass']].copy()
episodes_strict.to_parquet(OUTPUT_DIR / 'exercise_episodes_strict_recovery.parquet', index=False)
print(f'Saved exercise_episodes_strict_recovery.parquet  ({len(episodes_strict)} episodes, '
      f'{episodes_strict["participant_id"].nunique()} participants)')

# ---- summary CSV ----
if not episode_features_df.empty:
    grp_cols = ['detector_label', 'cadence_class', 'strain_class']
    grp_cols = [c for c in grp_cols if c in episode_features_df.columns]
    summary = (episode_features_df
               .groupby(grp_cols)[
                   ['duration_min', 'mean_HR_excess_stable',
                    'mean_steps_per_min', 'cadence_core_mean', 'cadence_core_p75',
                    'recovery_fraction_30', 'episode_confidence_score']
               ].median().round(2))
    summary.to_csv(OUTPUT_DIR / 'exercise_detection_summary.csv')
    print(f'Saved exercise_detection_summary.csv')

# ---- detector config ----
import json as _json
config = dict(
    run_id              = RUN_ID,
    STEP_RATE_FLOOR     = STEP_RATE_FLOOR,
    HR_EXCESS_FLOOR     = HR_EXCESS_FLOOR,
    MAX_BOUT_MIN        = MAX_BOUT_MIN,
    MIN_BOUT_MIN        = MIN_BOUT_MIN,
    ANCHOR_ON_MIN       = ANCHOR_ON_MIN,
    ANCHOR_OFF_MIN      = ANCHOR_OFF_MIN,
    MERGE_GAP_MIN       = MERGE_GAP_MIN,
    BACKTRACK_MAX_MIN   = BACKTRACK_MAX_MIN,
    HR_BASELINE_PERCENTILE = HR_BASELINE_PERCENTILE,
    HR_STRAIN_PERCENTILE   = HR_STRAIN_PERCENTILE,
    CADENCE_CORE_STEP_FLOOR = CADENCE_CORE_STEP_FLOOR,
    MIN_BASELINE_ROWS   = MIN_BASELINE_ROWS,
    cadence_bands       = dict(
        slow_walk  = [CADENCE_SLOW_WALK,  CADENCE_WALK],
        walk       = [CADENCE_WALK,       CADENCE_BRISK_WALK],
        brisk_walk = [CADENCE_BRISK_WALK, CADENCE_RUN_LIKE],
        run_like   = [CADENCE_RUN_LIKE,   9999],
    ),
    strain_bands        = dict(
        light    = [STRAIN_LIGHT,    STRAIN_MODERATE],
        moderate = [STRAIN_MODERATE, STRAIN_VIGOROUS],
        vigorous = [STRAIN_VIGOROUS, 9999],
    ),
    primary_definition  = 'clean = sleep<5% + duration in [10,120] + not long_active_envelope',
    strict_definition   = 'clean + recovery_pass (sensitivity subset)',
    recovery_min        = RECOVERY_MIN,
    recovery_frac_thresh = RECOVERY_FRAC_THRESH,
    long_active_hours   = LONG_ACTIVE_HOURS,
    glucose_rise_thresh_mg = GLUCOSE_RISE_THRESH_MG,
    conf_high_thresh     = CONF_HIGH_THRESH,
    conf_medium_thresh   = CONF_MEDIUM_THRESH,
    confidence_weights   = dict(q_activity=0.40, q_HR=0.30, q_RR=0.10,
                               q_duration=0.10, q_recovery=0.10,
                               q_sleep_penalty=-1.0),
    end_refinement       = dict(confirm_min=3,
                               max_extension_min=BACKTRACK_MAX_MIN,
                               sleep_safe=True),
    hr_floor_recommended = HR_EXCESS_FLOOR,
    gate_note           = (
        "Detection: HR_excess_stable (5th pct floor). "
        "Strain grading: HR_excess_strain (20th pct floor). "
        "Cadence grading: active-core cadence (minutes with steps >= CADENCE_CORE_STEP_FLOOR). "
        "No glucose, calories, or intensity_score in any gate."
    ),
)
with open(OUTPUT_DIR / 'detector_config.json', 'w') as _f:
    _json.dump(config, _f, indent=2)
print('Saved detector_config.json')


Saved exercise_anchor_positives.parquet  (63,665 rows)
Saved exercise_episodes_all.parquet  (3131 episodes)
Saved exercise_episodes_features.parquet  (3131 rows)
Saved exercise_episodes_clean.parquet  (2171 episodes, 788 participants)
Saved exercise_episodes_primary.parquet  (2171 episodes)
Saved exercise_episodes_strict_recovery.parquet  (1015 episodes, 574 participants)
Saved exercise_detection_summary.csv
Saved detector_config.json


In [75]:
print('\n=== FINAL RECOMMENDATION ===')
print('(Counts reloaded from saved files -- not from in-memory variables)')
print()

# Always reload from disk so the recommendation matches what was actually saved
_primary_path = OUTPUT_DIR / 'exercise_episodes_primary.parquet'
_strict_path  = OUTPUT_DIR / 'exercise_episodes_strict_recovery.parquet'
_all_path     = OUTPUT_DIR / 'exercise_episodes_features.parquet'

if not _primary_path.exists():
    print('[FAIL] exercise_episodes_primary.parquet not found -- run save cell first.')
else:
    _primary = pd.read_parquet(_primary_path)
    _strict  = pd.read_parquet(_strict_path)  if _strict_path.exists()  else pd.DataFrame()
    _all_eps = pd.read_parquet(_all_path)      if _all_path.exists()     else pd.DataFrame()

    n_primary = len(_primary)
    n_pid     = _primary['participant_id'].nunique() if n_primary else 0
    n_strict  = len(_strict)
    n_strict_pid = _strict['participant_id'].nunique() if n_strict else 0
    pid_7067  = 7067
    n7        = int((_all_eps['participant_id'] == pid_7067).sum()) if not _all_eps.empty else 0
    n7p       = int((_primary['participant_id'] == pid_7067).sum()) if n_primary else 0
    med_dur   = float(_primary['duration_min'].median()) if n_primary else 0.0

    all_pass = (
        n_primary >= 100
        and n_pid  >= 50
        and 10 <= med_dur <= 120
        and n7 >= 3
    )

    if all_pass:
        print('[PROCEED] Primary episodes pass all acceptance checks.')
        print('  Cadence and duration inconsistencies resolved.')
        print('  Proceed to Stage 2 residual diagnostic.')
    else:
        print('[REVIEW] One or more checks failing:')
        if n_primary < 100:
            print(f'  Primary episodes {n_primary} < 100.')
        if n_pid < 50:
            print(f'  Participants {n_pid} < 50.')
        if not (10 <= med_dur <= 120):
            print(f'  Median duration {med_dur:.1f} min outside 10-120.')
        if n7 < 3:
            print(f'  Participant 7067: {n7} total episodes (expected >= 3).')

    print(f'\n  Clean (primary):         {n_primary} episodes, {n_pid} participants')
    print(f'  Strict (+ recovery_pass): {n_strict} episodes, {n_strict_pid} participants')
    print(f'  Participant 7067: {n7} total episodes | {n7p} clean')
    print(f'  Median duration (primary): {med_dur:.1f} min')
    if not _all_eps.empty and 'onset_shift_min' in _all_eps.columns:
        shift_med = float(_all_eps['onset_shift_min'].median())
        print(f'  Median onset shift (all, confidence - refined): {shift_med:.1f} min')
    max_dur = float(_primary['duration_min'].max()) if n_primary else 0
    pct_long = float((_primary['duration_min'] > 180).mean()) if n_primary else 0
    print(f'  Max duration (primary): {max_dur:.0f} min  |  pct > 180 min: {pct_long:.1%}')



=== FINAL RECOMMENDATION ===
(Counts reloaded from saved files -- not from in-memory variables)

[REVIEW] One or more checks failing:
  Participant 7067: 0 total episodes (expected >= 3).

  Clean (primary):         2171 episodes, 788 participants
  Strict (+ recovery_pass): 1015 episodes, 574 participants
  Participant 7067: 0 total episodes | 0 clean
  Median duration (primary): 90.0 min
  Median onset shift (all, confidence - refined): 30.0 min
  Max duration (primary): 120 min  |  pct > 180 min: 0.0%


In [76]:
# ---- Figure and output summary ----
print("=" * 65)
print("FIGURE AND OUTPUT SUMMARY")
print("=" * 65)

import os, fnmatch

fig_dir    = OUTPUT_DIR / 'figures_v3'
output_dir = OUTPUT_DIR

FIGURE_REGISTRY = [
    (
        'aligned_ambulatory_refined_start.pdf',
        'Figure A',
        'Aligned summary at refined_start_time: HR_excess_stable, steps, RR. '
        'Thin dotted trace shows rolling local baseline (plot only, not used in gate).',
        fig_dir,
    ),
    (
        'timeline_*.pdf',
        'Figure B',
        'Participant-day timelines (5 participants, participant 7067 first). '
        'Stable HR floor + step floor annotated. Cadence+strain label on episode band.',
        fig_dir,
    ),
    (
        'recovery_aligned_end.pdf',
        'Figure C',
        'HR_excess_stable aligned at episode end (60-min window, rest-only minutes).',
        fig_dir,
    ),
    (
        'cadence_strain_heatmap.pdf',
        'Figure D',
        'Cadence x strain heatmap: episode counts and participant counts per cell.',
        fig_dir,
    ),
]

OUTPUT_REGISTRY = [
    ('exercise_anchor_positives.parquet',
     'Per-minute anchor-positive rows (gate: steps_mean_10 >= STEP_RATE_FLOOR '
     'AND HR_excess_stable_mean_10 >= HR_EXCESS_FLOOR AND awake).'),
    ('exercise_episodes_all.parquet',
     'All detected episodes before QC. Includes confidence_start_time and refined_start_time.'),
    ('exercise_episodes_features.parquet',
     'Episode feature table: cadence_class, strain_class, recovery_pass, '
     'long_active_envelope, all timing fields.'),
    ('exercise_episodes_primary.parquet',
     'Primary episodes: recovery_pass=True, sleep_contamination < 5%%, '
     'duration <= MAX_BOUT_MIN.'),
    ('exercise_detection_summary.csv',
     'Median features grouped by detector_label, cadence_class, strain_class.'),
    ('detector_config.json',
     'All threshold constants: STEP_RATE_FLOOR, HR_EXCESS_FLOOR, MAX_BOUT_MIN, '
     'cadence bands, strain bands, recovery params.'),
]

# ---- figures ----
print(f"\nFigures -- {fig_dir}")
print("-" * 65)
for pattern, label, description, search_dir in FIGURE_REGISTRY:
    if '*' in pattern:
        matches = sorted(f for f in os.listdir(search_dir)
                         if fnmatch.fnmatch(f, pattern)) if search_dir.exists() else []
        paths = [search_dir / m for m in matches]
    else:
        paths = [search_dir / pattern]

    exists_str = ""
    for p in paths:
        exists_str += f"\n      {'[OK]' if p.exists() else '[MISSING]'} {p.name}"

    print(f"  {label}: {pattern}")
    print(f"    {description}")
    print(f"    {exists_str.strip() if exists_str.strip() else '[MISSING]'}")
    print()

# ---- data outputs ----
print(f"Data outputs -- {output_dir}")
print("-" * 65)
for fname, desc in OUTPUT_REGISTRY:
    p = output_dir / fname
    if p.exists():
        size_kb = p.stat().st_size / 1024
        status = f"[OK]  {size_kb:>7,.0f} KB"
    else:
        status = "[MISSING]          "
    print(f"  {status}  {fname}")
    print(f"             {desc}")
    print()

# ---- all PDFs on disk ----
if fig_dir.exists():
    all_figs = sorted(fig_dir.glob("*.pdf"))
    print(f"All PDFs in figures_v3/ ({len(all_figs)} file(s)):")
    for f in all_figs:
        print(f"  {f.name:<55s} {f.stat().st_size/1024:>6.0f} KB")


FIGURE AND OUTPUT SUMMARY

Figures -- outputs/exercise_episode_detection_v2/figures_v3
-----------------------------------------------------------------
  Figure A: aligned_ambulatory_refined_start.pdf
    Aligned summary at refined_start_time: HR_excess_stable, steps, RR. Thin dotted trace shows rolling local baseline (plot only, not used in gate).
    [OK] aligned_ambulatory_refined_start.pdf

  Figure B: timeline_*.pdf
    Participant-day timelines (5 participants, participant 7067 first). Stable HR floor + step floor annotated. Cadence+strain label on episode band.
    [OK] timeline_1023_20230904.pdf
      [OK] timeline_1024_20230901.pdf
      [OK] timeline_1029_20230910.pdf
      [OK] timeline_1030_20230910.pdf
      [OK] timeline_4022_20230930.pdf
      [OK] timeline_refined_1023_20230904.pdf
      [OK] timeline_refined_1024_20230901.pdf
      [OK] timeline_refined_1029_20230910.pdf
      [OK] timeline_refined_1030_20230910.pdf
      [OK] timeline_refined_4022_20230930.pdf

  Fig